<a href="https://colab.research.google.com/github/bitlabsdevteam/Detects-Implicit-Bias-in-LLM-Outputs-/blob/main/colab/Fairsteer_BAD_training_Mistral7B_Instruct_Quantisation_4bytes_zero_short_20251229_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup & Installation

In [ ]:
print(" Installing optimized stack for L4...\n")
# We use sdpa (built-in), so no need for flash-attn pip install
!pip install -q -U torch transformers accelerate bitsandbytes datasets huggingface_hub tqdm scikit-learn matplotlib seaborn pandas safetensors

print(" Installation complete!\n")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Change this in your Config Cell
local_save_dir = "/content/drive/MyDrive/FairSteer_Research_ZS_mistralai/Mistral-7B-Instruct-v0.3"

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
import json
import random
from tqdm.auto import tqdm
from datetime import datetime
from typing import Dict, List, Tuple, Optional
import os
import gc


from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from datasets import load_dataset
from huggingface_hub import HfApi, create_repo, login
from safetensors.torch import save_file


from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings


warnings.filterwarnings('ignore')


SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*80)
print(" BAD Classifier Training: Mistral-7B T4 Optimized Setup")
print("="*80)
print(f"Device: {device}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {total_mem:.2f} GB")

    # T4 Check: T4 supports FP16 but NOT BF16. Ampere (A100/A10) supports BF16.
    # We must ensure we don't try to use BF16 on a T4.
    if torch.cuda.is_bf16_supported():
        print("Precision: BFloat16 (Supported)")
        compute_dtype = torch.bfloat16
    else:
        print("Precision: Float16 (T4 Standard)")
        compute_dtype = torch.float16
else:
    print("WARNING: No GPU detected. Mistral 7B will likely crash or be extremely slow.")
    compute_dtype = torch.float32

print("="*80 + "\n")

## 2. Configuration

In [ ]:


import torch

class TrainingConfig:

    # --- MODEL & DATA ---
    base_model_name = "mistralai/Mistral-7B-Instruct-v0.3"

    # Mistral 7B Hidden Dimension
    model_hidden_dim = 4096

    # Zero-shot is short, but we allow 512 to handle edge cases safely.
    # This keeps the KV cache memory footprint predictable.
    max_length = 512

    # Dataset Paths
    bbq_dataset_name = "bitlabsdb/BBQ_dataset"
    bbq_target_loc_dataset = "bitlabsdb/bbq_target_loc_dedup"

    num_bbq_samples = 58492
    train_val_split = 0.8

    # --- BALANCING STRATEGY ---
    use_balanced_sampling = True
    target_samples_per_class = None
    min_samples_per_class = 5000
    SEED = 42


    batch_size = 2048

    learning_rate = 1e-3
    num_epochs = 50
    dropout_rate = 0.1
    weight_decay = 1e-4 # Helps prevent overfitting
    early_stopping_patience = 15
    gradient_clip_norm = 1.0

    # Run all layer for initial evaluation
    candidate_layers_range = list(range(0, 31))

    # --- LABELS ---
    LABEL_BIASED = 0
    LABEL_UNBIASED = 1

    # --- DEPLOYMENT PATHS ---
    hf_repo_name = "bitlabsdb/bad-classifier-mistral-7b-fairsteer-zs-Instruct-v0.3"
    hf_private = False
    local_save_dir = "./bad_model_fairsteer_mistral_7b"

config = TrainingConfig()

print("="*80)
print(" 🛡️ CONFIGURATION UPDATED: MISTRAL-7B (STABLE PRODUCTION)")
print("="*80)
print(f"   • Model:           {config.base_model_name}")
print(f"   • Extraction Batch:{config.extraction_batch_size} (Safe Buffer: ~12GB)")
print(f"   • Layers Scanning: {config.candidate_layers_range}")
print(f"   • Save Location:   {config.local_save_dir}")
print("="*80 + "\n")

## 3. Model Definition
### A simple single linear and single droput linear model.
#### No over-engineered and overkilled the dataset

In [ ]:
class BADClassifier(nn.Module):
    """
    Biased Activation Detection (BAD) Classifier - FairSteer Aligned

    Architecture: Dropout -> Linear layer -> Sigmoid

    """

    def __init__(self, input_dim: int, dropout_rate=config.dropout_rate):
        super().__init__()

        # dropout layer
        self.dropout = nn.Dropout(p=dropout_rate)

        # 2. Linear Layer
        self.linear = nn.Linear(input_dim, 1)


        # Xavier initialization for stable training
        nn.init.xavier_uniform_(self.linear.weight)
        nn.init.zeros_(self.linear.bias)

    def forward(self, x):
        """
        Forward pass: Dropout -> Linear -> Logits
        """
        # Apply dropout first
        x = self.dropout(x)
        return self.linear(x)

    def predict_proba(self, x):
        """
        Get probability of being UNBIASED
        Returns: p(y=1) where y=1 means UNBIASED
        """
        logits = self.forward(x)
        probs = torch.sigmoid(logits).squeeze(-1)
        return probs

    def detect_bias(self, x, threshold: float = 0.05):
        """
        Detect biased activations using FairSteer threshold
        """
        unbiased_prob = self.predict_proba(x)
        is_biased = unbiased_prob < threshold
        return is_biased, unbiased_prob

print("✅ BAD Classifier defined (Architecture: Dropout -> Linear + Sigmoid)\n")

## 4. Data Ingestion & Integrity Analysis: Merging BBQ with Targets

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from datasets import load_dataset
import warnings


def load_and_merge_bbq(config: TrainingConfig) -> pd.DataFrame:
    """
    Loads BBQ and Targets, merges them, and visualizes both
    the Successful Joins and the Missing/Dropped records.
    """

    print("="*80)
    print(" 🚀 SUPER EFFECTIVE BBQ LOADER & MERGER (WITH LOSS ANALYSIS)")
    print("="*80 + "\n")

    # 1. LOAD BBQ
    print("1. Loading BBQ Dataset...")
    try:
        bbq_ds = load_dataset(config.bbq_dataset_name, split="train")
    except:
        print("BBQ dataset loading failed...")

    df_bbq = pd.DataFrame(bbq_ds)

    # Force ID to int
    df_bbq['example_id'] = pd.to_numeric(df_bbq['example_id'], errors='coerce').fillna(-1).astype(int)

    # 2. LOAD TARGETS
    print("2. Loading Target Locations...")
    loc_ds = load_dataset(config.bbq_target_loc_dataset, split="train")
    df_loc = pd.DataFrame(loc_ds)


    # 1. Clean Example ID
    df_loc['example_id'] = pd.to_numeric(df_loc['example_id'], errors='coerce')
    df_loc = df_loc.dropna(subset=['example_id'])
    df_loc['example_id'] = df_loc['example_id'].astype(int)

    # 2. Clean Target Loc
    df_loc['target_loc'] = pd.to_numeric(df_loc['target_loc'], errors='coerce')

    # 3. Filter Valid Rows (0, 1, 2)
    df_loc = df_loc[df_loc['target_loc'].isin([0, 1, 2])]

    # 4. Cast ONLY the target column to int (Prevents the 'Race_x_gender' error)
    df_loc['target_loc'] = df_loc['target_loc'].astype(int)

    # 5. Deduplicate
    df_loc = df_loc.drop_duplicates(subset=['example_id'], keep='first')

    # 3. MERGE & IDENTIFY MISSING DATA
    print("3. Merging & Analyzing Integrity...")

    # Left Join to find what is missing
    integrity_check = pd.merge(
        df_bbq,
        df_loc[['example_id', 'target_loc']],
        on='example_id',
        how='left',
        indicator=True
    )

    # Split into Keep vs Drop
    df_merged = integrity_check[integrity_check['_merge'] == 'both'].drop(columns=['_merge']).copy()
    df_missing = integrity_check[integrity_check['_merge'] == 'left_only'].copy()

    # Final type fix
    df_merged['target_loc'] = df_merged['target_loc'].astype(int)

    count_total = len(df_bbq)
    count_kept = len(df_merged)
    count_lost = len(df_missing)

    print(f"   ✅ Merge Complete.")
    print(f"   - Total Questions: {count_total:,}")
    print(f"   - Valid Training Samples: {count_kept:,} ({(count_kept/count_total):.1%})")
    print(f"   - Missing/Dropped: {count_lost:,} ({(count_lost/count_total):.1%})")

    # 4. VISUALIZATION DASHBOARD
    print("\n📊 Generating Integrity Dashboard...")
    sns.set_theme(style="whitegrid")

    # Setup a 1x3 grid
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    plt.suptitle(f"Data Integrity Analysis (N={count_total:,})", fontsize=16, weight='bold')

    # --- Plot 1: Overall Retention ---
    sns.barplot(
        x=['Kept (Merged)', 'Lost (No Target)'],
        y=[count_kept, count_lost],
        palette=['green', 'red'],
        ax=axes[0]
    )
    axes[0].set_title("Total Data Retention", fontsize=14)
    axes[0].bar_label(axes[0].containers[0])
    axes[0].set_ylabel("Number of Samples")

    # --- Plot 2: Valid Data Distribution (By Category) ---
    if not df_merged.empty and 'category' in df_merged.columns:
        sns.countplot(
            y='category',
            data=df_merged,
            order=df_merged['category'].value_counts().index,
            palette="viridis",
            ax=axes[1]
        )
        axes[1].set_title(f"✅ Valid Training Data ({count_kept:,})", fontsize=14)
        axes[1].set_xlabel("Count")
        axes[1].set_ylabel("")

    # --- Plot 3: Missing Data Distribution (By Category) ---
    if not df_missing.empty and 'category' in df_missing.columns:
        sns.countplot(
            y='category',
            data=df_missing,
            order=df_missing['category'].value_counts().index,
            palette="Reds_r",
            ax=axes[2]
        )
        axes[2].set_title(f"❌ Missing/Dropped Records ({count_lost:,})", fontsize=14)
        axes[2].set_xlabel("Count")
        axes[2].set_ylabel("")
    else:
        axes[2].text(0.5, 0.5, "No Missing Data! 🎉", ha='center', fontsize=14)

    plt.tight_layout()
    plt.show()

    print("="*80 + "\n")
    return df_merged


df_final = load_and_merge_bbq(config)
df_final.head(5)



#  5. Create Unique Groups
## Grouping BBQ databse with category and Question Index



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cell Title
## 2. Feature Engineering: Group Identification for Leakage Prevention

print("="*80)
print(" 🛡️ PREVENTING DATA LEAKAGE: GROUP IDENTIFICATION")
print("="*80 + "\n")

# ---------------------------------------------------------
# 1. INPUT VALIDATION (Fail Fast Principle)
# ---------------------------------------------------------
# We expect 'df_final' from the previous step.
# We do not guess variable names; we enforce the pipeline structure.

if 'df_final' not in globals():
    raise ValueError("❌ Pipeline Error: 'df_final' is missing. Please run Step 1 (Loader) first.")

df_grouped = df_final.copy()
print(f"   ✅ Input Data Loaded. Shape: {len(df_grouped):,} rows.")

# ---------------------------------------------------------
# 2. DETERMINISTIC GROUPING LOGIC
# ---------------------------------------------------------
print("\n2. Generating Robust Group IDs...")

# We define a group strictly by 'category' and 'question_index'.
# All variations (Ambiguous/Disambiguated) of the same template MUST share this ID.
required_cols = ['category', 'question_index']

if not all(col in df_grouped.columns for col in required_cols):
    raise ValueError(f"❌ Missing required columns for grouping: {required_cols}")

# Create a human-readable Group Name (for debugging)
df_grouped['group_name'] = df_grouped['category'].astype(str) + "-" + df_grouped['question_index'].astype(str)

# Create a Machine-Readable Group ID (Deterministic Integer)
# ngroup() assigns a unique integer (0 to N) to each unique combination.
# This is safer than hash() because it never collides and is reproducible.
df_grouped['group_id'] = df_grouped.groupby(required_cols).ngroup()

# ---------------------------------------------------------
# 3. LEAKAGE SAFETY CHECK
# ---------------------------------------------------------
print("\n3. Verifying Leakage Protection...")

# Verify that pairs are actually grouped together
group_counts = df_grouped['group_id'].value_counts()
paired_groups = (group_counts >= 2).sum()
single_groups = (group_counts == 1).sum()

print(f"   - Total Unique Groups: {df_grouped['group_id'].nunique():,}")
print(f"   - Groups with Pairs/Tuples: {paired_groups:,} (Safe for split)")
print(f"   - Groups with Singletons: {single_groups:,} (Data missing?)")

if single_groups > 0:
    print("   ⚠️ WARNING: Some groups have only 1 sample. This implies missing Ambiguous/Disambiguated counterparts.")

# ---------------------------------------------------------
# 4. VISUALIZATION
# ---------------------------------------------------------
print("\n📊 Generating Grouping Analytics...")
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plt.subplots_adjust(wspace=0.3)

# Plot 1: Group Size Distribution
# We want to see '2' as the dominant bar (meaning pairs exist)
sizes = df_grouped.groupby('group_id').size()
sns.histplot(sizes, bins=range(1, 6), discrete=True, color='teal', ax=axes[0])
axes[0].set_title("Group Size Distribution (Target: Pairs of 2)", fontsize=14)
axes[0].set_xlabel("Samples per Group")
axes[0].set_ylabel("Frequency")
axes[0].set_xticks([1, 2, 3, 4])

# Plot 2: Groups per Category
cat_counts = df_grouped.groupby('category')['group_id'].nunique().sort_values(ascending=False)
sns.barplot(x=cat_counts.values, y=cat_counts.index, palette="viridis", ax=axes[1])
axes[1].set_title("Unique Template Groups per Category", fontsize=14)
axes[1].set_xlabel("Number of Unique Question Templates")

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 5. PIPELINE EXPORT
# ---------------------------------------------------------
# Update the main variable for the next step (Model Training)
bbq_merged_df = df_grouped

print(f"✅ 'bbq_merged_df' is ready for splitting.")
print(f"   Use 'group_id' in GroupShuffleSplit to prevent leakage.")
print("="*80 + "\n")

# 6. Evaluate the grouping logic is correctly built

In [ ]:
# Cell Title
## 2. Feature Engineering: Group Identification for Leakage Prevention

print("="*80)
print(" 🛡️ PREVENTING DATA LEAKAGE: GROUP IDENTIFICATION")
print("="*80 + "\n")

# ---------------------------------------------------------
# 1. INPUT VALIDATION
# ---------------------------------------------------------
if 'df_final' not in globals():
    raise ValueError("❌ Pipeline Error: 'df_final' is missing. Please run Step 1 (Loader) first.")

df_grouped = df_final.copy()
print(f"   ✅ Input Data Loaded. Shape: {len(df_grouped):,} rows.")

# ---------------------------------------------------------
# 2. DETERMINISTIC GROUPING LOGIC
# ---------------------------------------------------------
print("\n2. Generating Robust Group IDs...")

required_cols = ['category', 'question_index']

if not all(col in df_grouped.columns for col in required_cols):
    raise ValueError(f"❌ Missing required columns for grouping: {required_cols}")

# Human-readable group name
df_grouped['group_name'] = (
    df_grouped['category'].astype(str) + "-" +
    df_grouped['question_index'].astype(str)
)

# Machine-readable group ID (keeps all 4 variants together)
df_grouped['group_id'] = df_grouped.groupby(required_cols).ngroup()

# ---------------------------------------------------------
# 3. LEAKAGE SAFETY CHECK
# ---------------------------------------------------------
print("\n3. Verifying Leakage Protection...")

# Check group sizes
group_counts = df_grouped['group_id'].value_counts()
paired_groups = (group_counts >= 2).sum()
single_groups = (group_counts == 1).sum()

print(f"   - Total Unique Groups: {df_grouped['group_id'].nunique():,}")
print(f"   - Groups with 2+ samples: {paired_groups:,}")
print(f"   - Groups with 1 sample: {single_groups:,}")

# ✅ NEW: Verify 4-variant structure
if 'context_condition' in df_grouped.columns and 'question_polarity' in df_grouped.columns:
    print("\n4. Verifying BBQ 4-Variant Structure...")

    # Count how many groups have all 4 variants
    def count_variants(group_df):
        return len(group_df)

    variant_counts = df_grouped.groupby('group_id').apply(count_variants)

    groups_with_4 = (variant_counts == 4).sum()
    groups_with_2 = (variant_counts == 2).sum()
    groups_with_1 = (variant_counts == 1).sum()

    print(f"   - Groups with 4 variants (ideal): {groups_with_4:,}")
    print(f"   - Groups with 2 variants: {groups_with_2:,}")
    print(f"   - Groups with 1 variant: {groups_with_1:,}")

    if groups_with_4 > 0:
        pct_complete = (groups_with_4 / df_grouped['group_id'].nunique()) * 100
        print(f"   - Complete templates: {pct_complete:.1f}%")

    # Show breakdown for a sample group
    sample_group_id = df_grouped['group_id'].iloc[0]
    sample_variants = df_grouped[df_grouped['group_id'] == sample_group_id][
        ['category', 'question_index', 'context_condition', 'question_polarity', 'group_id']
    ].drop_duplicates()

    print(f"\n   Sample Group (ID={sample_group_id}):")
    print(sample_variants.to_string(index=False))

if single_groups > 0:
    print("\n   ⚠️ NOTE: Some groups have <4 samples (incomplete templates).")
    print("   This is normal for BBQ - these samples are still usable.")
    print("   They will be kept in training/validation.")

# ---------------------------------------------------------
# 5. VISUALIZATION
# ---------------------------------------------------------
print("\n📊 Generating Grouping Analytics...")
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plt.subplots_adjust(wspace=0.3)

# Plot 1: Group Size Distribution
sizes = df_grouped.groupby('group_id').size()
sns.histplot(sizes, bins=range(1, 6), discrete=True, color='teal', ax=axes[0])
axes[0].set_title("Group Size Distribution\n(Target: 4 variants per template)", fontsize=14)
axes[0].set_xlabel("Samples per Group")
axes[0].set_ylabel("Frequency")
axes[0].set_xticks([1, 2, 3, 4, 5])
axes[0].axvline(x=4, color='red', linestyle='--', linewidth=2, label='Ideal (4 variants)')
axes[0].legend()

# Plot 2: Groups per Category
cat_counts = df_grouped.groupby('category')['group_id'].nunique().sort_values(ascending=False)
sns.barplot(x=cat_counts.values, y=cat_counts.index, palette="viridis", ax=axes[1])
axes[1].set_title("Unique Template Groups per Category", fontsize=14)
axes[1].set_xlabel("Number of Unique Question Templates")

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 6. PIPELINE EXPORT
# ---------------------------------------------------------
bbq_merged_df = df_grouped

print(f"\n✅ 'bbq_merged_df' is ready for splitting.")
print(f"   Grouping Strategy: {required_cols}")
print(f"   Leakage Prevention: All variants of same template stay together")
print(f"   Use 'group_id' in GroupShuffleSplit to ensure no leakage.")
print("="*80 + "\n")

# 8. Activation Extraction: Batched Inference with Strict Bias Detection


In [ ]:
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from typing import Tuple

# ==============================================================================
# FORWARD HOOK: RESIDUAL STREAM CAPTURE
# ==============================================================================
class LayerActivationHook:
    """
    Captures last-token residual stream activation at layer l.

    Methodology:
    - Extracts x_l (block output after layer l)
    - This is the residual stream state carried forward
    - Consistent with FairSteer BAD formulation
    - Moves to CPU immediately to prevent VRAM spikes
    """
    def __init__(self):
        self.captured_tensor = None

    def __call__(self, module, input, output):
        # Handle tuple output (hidden_states, past_key_values)
        hidden_states = output[0] if isinstance(output, tuple) else output

        # Extract last token: [Batch, Seq, Dim] -> [Batch, Dim]
        # This is a_l in BAD paper: residual stream activation at layer l
        self.captured_tensor = hidden_states[:, -1, :].detach().cpu()

# ==============================================================================
# MAIN EXTRACTION FUNCTION
# ==============================================================================
def extract_bbq_activations(
    model,
    tokenizer,
    merged_df: pd.DataFrame,
    config,
    layer_idx: int,
    batch_size: int = 16,
    debug_limit: int = 5
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Extracts residual stream activations for BAD classifier training.

    Architecture:
    - Hooks into model.model.layers[layer_idx]
    - Captures last-token state after layer processing
    - Batched inference for GPU efficiency

    Prompt Format:
    - Mistral-7B-Instruct with [INST] tags
    - Format: [INST] {sys_msg}\n\n{content} [/INST] Answer:
    - Last token is ":" (colon) - the prediction trigger

    Labeling Logic:
    - Ambiguous: target=biased(0), unknown=unbiased(1)
    - Disambiguated: override_fact=biased(0), follow_fact=unbiased(1)

    Returns:
    - activations: [N, hidden_dim] residual stream states
    - labels: [N] binary labels (0=biased, 1=unbiased)
    - groups: [N] group IDs for leakage prevention
    """

    # ----------------------------------------------------------------
    # 1. SETUP
    # ----------------------------------------------------------------
    model.eval()
    device = next(model.parameters()).device

    if tokenizer.padding_side != "left":
        raise ValueError("❌ Tokenizer must use left padding for batch inference")

    print(f"🚀 Layer {layer_idx} | Device: {device} | Batch: {batch_size}")

    # ----------------------------------------------------------------
    # 2. REGISTER HOOK
    # ----------------------------------------------------------------
    if not hasattr(model, "model") or not hasattr(model.model, "layers"):
        raise ValueError("❌ Model architecture mismatch: Cannot find 'model.model.layers'")

    hook_helper = LayerActivationHook()
    handle = model.model.layers[layer_idx].register_forward_hook(hook_helper)

    # ----------------------------------------------------------------
    # 3. PROMPT PREPROCESSING
    # ----------------------------------------------------------------
    print("📝 Preprocessing prompts...")

    # Vocab map for A/B/C logits
    vocab_map = {
        'A': tokenizer.encode('A', add_special_tokens=False)[-1],
        'B': tokenizer.encode('B', add_special_tokens=False)[-1],
        'C': tokenizer.encode('C', add_special_tokens=False)[-1]
    }

    # Keywords for "unknown" answer detection
    neutral_keywords = {
        "unknown", "can't", "cannot", "not be determined", "not known",
        "undetermined", "ambiguous", "not enough", "insufficient",
        "neither", "none", "don't know", "unable to determine"
    }

    prompts = []
    metadata = []

    for _, row in merged_df.iterrows():
        choices = [
            str(row.get('ans0', '')),
            str(row.get('ans1', '')),
            str(row.get('ans2', ''))
        ]

        # Identify "unknown" option
        unknown_idx = -1
        for i, choice in enumerate(choices):
            if any(kw in choice.lower() for kw in neutral_keywords):
                unknown_idx = i
                break

        # Skip ambiguous samples without unknown option
        if row['context_condition'] == 'ambig' and unknown_idx == -1:
            continue

        # Build content (WITHOUT "Answer:" yet)
        content = (
            f"{row['context']} {row['question']}\n"
            f"A. {choices[0]}\n"
            f"B. {choices[1]}\n"
            f"C. {choices[2]}"
        )

        # Apply Mistral instruction template
        sys_msg = "Select the single best answer."
        inst_block = tokenizer.apply_chat_template(
            [{"role": "user", "content": f"{sys_msg}\n\n{content}"}],
            tokenize=False,
            add_generation_prompt=True
        )

        # Append trigger OUTSIDE [INST] block
        full_prompt = inst_block + " Answer:"

        # Store
        prompts.append(full_prompt)
        metadata.append({
            'target_loc': int(row['target_loc']),
            'unknown_idx': unknown_idx,
            'context_condition': row['context_condition'],
            'correct_label': int(row.get('label', -1)),
            'group_id': row['group_id']
        })

    print(f"✅ {len(prompts):,} valid samples ready")

    # ----------------------------------------------------------------
    # 4. BATCHED EXTRACTION
    # ----------------------------------------------------------------
    final_activations = []
    final_labels = []
    final_groups = []
    stats = {'biased': 0, 'unbiased': 0, 'skipped': 0}
    debug_counter = 0

    try:
        for i in tqdm(range(0, len(prompts), batch_size), desc=f"⚡ Layer {layer_idx}"):
            batch_prompts = prompts[i : i + batch_size]
            batch_meta = metadata[i : i + batch_size]

            # Tokenize (template already added <s> BOS token)
            inputs = tokenizer(
                batch_prompts,
                return_tensors="pt",
                max_length=getattr(config, 'max_length', 512),
                truncation=True,
                padding=True,
                add_special_tokens=False
            ).to(device)

            # Forward pass (hook captures activations automatically)
            with torch.inference_mode():
                outputs = model(**inputs)

            # Get logits and activations
            logits = outputs.logits[:, -1, :]  # Last token logits
            probs = torch.softmax(logits, dim=-1)
            batch_acts = hook_helper.captured_tensor.float().numpy()  # From hook

            # Process each sample in batch
            for j, meta in enumerate(batch_meta):
                # Get probabilities for A/B/C
                p_a = probs[j, vocab_map['A']].item()
                p_b = probs[j, vocab_map['B']].item()
                p_c = probs[j, vocab_map['C']].item()
                pred_idx = np.argmax([p_a, p_b, p_c])

                # Extract metadata
                tgt = meta['target_loc']
                unk = meta['unknown_idx']
                cond = meta['context_condition']
                correct = meta['correct_label']

                # Determine label
                label = -1

                if cond == 'ambig':
                    if pred_idx == tgt:
                        label = config.LABEL_BIASED
                        stats['biased'] += 1
                    elif pred_idx == unk:
                        label = config.LABEL_UNBIASED
                        stats['unbiased'] += 1
                    else:
                        stats['skipped'] += 1

                elif cond == 'disambig':
                    if correct != -1:
                        # Bias overrides fact
                        if pred_idx == tgt and pred_idx != correct:
                            label = config.LABEL_BIASED
                            stats['biased'] += 1
                        # Fact overrides bias
                        elif pred_idx == correct and correct != tgt:
                            label = config.LABEL_UNBIASED
                            stats['unbiased'] += 1
                        else:
                            stats['skipped'] += 1
                    else:
                        stats['skipped'] += 1

                # Debug output
                if debug_counter < debug_limit and label != -1:
                    print(f"\n🔍 DEBUG #{debug_counter+1}")
                    print(f"   Condition: {cond}")
                    print(f"   Prediction: {['A','B','C'][pred_idx]}")
                    print(f"   Target: {tgt} | Unknown: {unk} | Correct: {correct}")
                    print(f"   Label: {label}")
                    debug_counter += 1

                # Store if valid
                if label != -1:
                    final_activations.append(batch_acts[j])
                    final_labels.append(label)
                    final_groups.append(meta['group_id'])

            # Clear hook buffer
            hook_helper.captured_tensor = None

    finally:
        # Always cleanup
        handle.remove()
        torch.cuda.empty_cache()

    # ----------------------------------------------------------------
    # 5. SUMMARY
    # ----------------------------------------------------------------
    print("\n" + "="*60)
    print("📊 EXTRACTION COMPLETE")
    print("="*60)
    print(f"   🔴 Biased:   {stats['biased']:,}")
    print(f"   🟢 Unbiased: {stats['unbiased']:,}")
    print(f"   ⏭️  Skipped:  {stats['skipped']:,}")
    print(f"   💾 Total:    {len(final_labels):,}")
    print("="*60 + "\n")

    return (
        np.array(final_activations),
        np.array(final_labels),
        np.array(final_groups)
    )

print("✅ Extraction Function Ready")

# 9.  Activation Extraction: Batched Inference with Strict Bias Detection - PCA Chart


In [ ]:
# ==========================================
# CELL 8: DATA PROCESSING PIPELINE (ROBUST)
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.decomposition import PCA

# =========================================================
# 1. ADVANCED VISUALIZATION SUITE
# =========================================================
def visualize_data_health(X_train, y_train, X_val, y_val, X_train_raw_sample, X_train_scaled_sample):
    """
    Comprehensive Data Health Check:
    1. Balance Check (Train vs Val)
    2. Scaling Effect (Global Distribution)
    3. Linear Separability Check (PCA - Critical for FairSteer)
    """
    sns.set_theme(style="whitegrid")
    fig = plt.figure(figsize=(20, 10))
    gs = fig.add_gridspec(2, 3)

    # --- Plot 1: Class Balance (Train) ---
    ax1 = fig.add_subplot(gs[0, 0])
    sns.countplot(x=y_train, ax=ax1, palette='Blues')
    ax1.set_title(f"Train Balance (N={len(y_train):,})", fontweight='bold')
    ax1.set_xticklabels(['Biased (0)', 'Unbiased (1)'])
    try:
        ax1.bar_label(ax1.containers[0])
    except: pass # Safety for older matplotlib versions

    # --- Plot 2: Class Balance (Val) ---
    ax2 = fig.add_subplot(gs[0, 1])
    sns.countplot(x=y_val, ax=ax2, palette='Greens')
    ax2.set_title(f"Val Balance (N={len(y_val):,})", fontweight='bold')
    ax2.set_xticklabels(['Biased (0)', 'Unbiased (1)'])
    try:
        ax2.bar_label(ax2.containers[0])
    except: pass

    # --- Plot 3: Scaling Effect (Global Histogram) ---
    ax3 = fig.add_subplot(gs[0, 2])

    # Subsample for speed (1000 samples * 100 features)
    if len(X_train_raw_sample) > 0:
        subset_idx = np.random.choice(len(X_train_raw_sample), min(1000, len(X_train_raw_sample)), replace=False)
        # Flatten first 100 dims to visualize distribution
        raw_flat = X_train_raw_sample[subset_idx, :100].flatten()
        scaled_flat = X_train_scaled_sample[subset_idx, :100].flatten()

        sns.kdeplot(raw_flat, ax=ax3, color='red', fill=True, label='Raw (High Var)', alpha=0.3)
        sns.kdeplot(scaled_flat, ax=ax3, color='blue', fill=True, label='Scaled (N(0,1))', alpha=0.3)
        ax3.set_title("Feature Scaling Effect (Global)", fontweight='bold')
        ax3.set_xlim(-5, 5)
        ax3.legend()

    # --- Plot 4: The FairSteer Hypothesis Check (PCA) ---
    ax4 = fig.add_subplot(gs[1, :])

    if len(X_train) > 5: # PCA requires at least a few samples
        print("   ...computing PCA for visualization (CPU)...")
        # RandomizedPCA is extremely efficient on CPU for high dims
        pca = PCA(n_components=2, svd_solver='randomized', random_state=42)

        # Plot max 2000 points to avoid clutter
        plot_n = min(2000, len(X_train))
        plot_idx = np.random.choice(len(X_train), plot_n, replace=False)

        X_pca = pca.fit_transform(X_train[plot_idx])
        y_pca = y_train[plot_idx]

        scatter = ax4.scatter(X_pca[:, 0], X_pca[:, 1], c=y_pca, cmap='coolwarm', alpha=0.6, edgecolor='w', s=40)
        ax4.set_title(f"Linear Separability Check (PCA Projection of {plot_n} samples)", fontweight='bold', fontsize=14)
        ax4.set_xlabel("PC 1")
        ax4.set_ylabel("PC 2")

        handles, _ = scatter.legend_elements()
        ax4.legend(handles, ['Biased', 'Unbiased'], loc="upper right", title="Class")
    else:
        ax4.text(0.5, 0.5, "Not enough data for PCA", ha='center', fontsize=14)

    plt.tight_layout()
    plt.show()

# =========================================================
# 2. OPTIMIZED PIPELINE
# =========================================================
def prepare_data_pipeline(activations, labels, group_ids, config):
    """
    1. Leakage-Proof Split (GroupShuffleSplit)
    2. RAM-Optimized Balancing (Index shuffling)
    3. Robust Standardization (Float32 + NaN check)
    4. FairSteer Visualization
    """
    print("\n" + "="*80)
    print("  🔧 DATA PIPELINE EXECUTION (Optimized)")
    print("="*80)

    # A. Safety Check for NaNs
    if np.isnan(activations).any():
        print("  ⚠️ WARNING: NaNs detected in raw activations. Replacing with 0.")
        activations = np.nan_to_num(activations)

    # ---------------------------------------------------------
    # 1. Group-wise Split (Prevent Leakage)
    # ---------------------------------------------------------
    gss = GroupShuffleSplit(n_splits=1, train_size=config.train_val_split, random_state=config.SEED)
    try:
        train_idx, val_idx = next(gss.split(activations, labels, groups=group_ids))
    except StopIteration:
        raise ValueError("❌ Split failed. Dataset likely too small or groups not defined correctly.")

    # Verification
    train_groups = set(group_ids[train_idx])
    val_groups = set(group_ids[val_idx])
    overlap = train_groups.intersection(val_groups)

    print(f"  Step 1: Split")
    print(f"     Train Indices: {len(train_idx):,}")
    print(f"     Val Indices:   {len(val_idx):,}")

    if len(overlap) > 0:
        raise RuntimeError(f"❌ CRITICAL LEAKAGE: {len(overlap)} groups exist in both sets!")
    else:
        print(f"     ✅ Leakage Check Passed.")

    # ---------------------------------------------------------
    # 2. Memory-Optimized Balancing (Index-Based)
    # ---------------------------------------------------------
    def get_balanced_indices(full_indices, full_labels, target_n=None):
        curr_y = full_labels[full_indices]
        idx_biased = np.where(curr_y == config.LABEL_BIASED)[0]
        idx_unbiased = np.where(curr_y == config.LABEL_UNBIASED)[0]

        n_samples = min(len(idx_biased), len(idx_unbiased))

        if n_samples == 0:
             # Safety guard for empty classes
             print(f"  ⚠️ WARNING: One class has 0 samples! (Biased: {len(idx_biased)}, Unbiased: {len(idx_unbiased)})")
             return np.array([], dtype=int)

        if target_n:
            n_samples = min(n_samples, target_n)

        # Select relative indices
        keep_b = np.random.choice(idx_biased, n_samples, replace=False)
        keep_u = np.random.choice(idx_unbiased, n_samples, replace=False)

        # Map back to global indices
        global_indices = np.concatenate([full_indices[keep_b], full_indices[keep_u]])
        np.random.shuffle(global_indices)
        return global_indices

    print(f"  Step 2: Balancing (Index-Based Optimization)...")
    train_idx_bal = get_balanced_indices(train_idx, labels, config.target_samples_per_class)
    val_idx_bal = get_balanced_indices(val_idx, labels, config.target_samples_per_class)

    if len(train_idx_bal) == 0:
        raise ValueError("❌ Training Set is empty after balancing! Check your label distribution.")

    # Slice Arrays (Memory Spike happens only here)
    X_train = activations[train_idx_bal]
    y_train = labels[train_idx_bal]
    X_val = activations[val_idx_bal]
    y_val = labels[val_idx_bal]

    # ---------------------------------------------------------
    # 3. Standardization (Float32)
    # ---------------------------------------------------------
    # Casting to float32 guarantees compatibility with PyTorch/TensorFlow
    # and saves 50% RAM vs float64. Accuracy is preserved (LLMs are fp16).
    X_train = X_train.astype(np.float32)
    X_val = X_val.astype(np.float32)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    print(f"  Step 3: Standardized")
    print(f"     Train Mean: {np.mean(X_train_scaled):.4f} | Std: {np.std(X_train_scaled):.4f}")

    # ---------------------------------------------------------
    # 4. FairSteer Visualization
    # ---------------------------------------------------------
    print("  Step 4: Visualizing Data Health...")
    visualize_data_health(X_train_scaled, y_train, X_val_scaled, y_val, X_train, X_train_scaled)

    return X_train_scaled, y_train, X_val_scaled, y_val, scaler

print("✅ Data Pipeline defined (RAM-Optimized + PCA Check)\n")

# 11. Multi-layer extraction with hook

In [ ]:


import os
import shutil
import pickle
import torch
import numpy as np
import gc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm.auto import tqdm

print("="*80)
print(" 📉 MULTI-LAYER EXTRACTION ENGINE (WITH LIVE MONITORING)")
print("="*80 + "\n")

# ---------------------------------------------------------
# 1. SETUP & PATHS
# ---------------------------------------------------------
abs_save_dir = os.path.abspath(config.local_save_dir)
cache_dir = os.path.join(abs_save_dir, "cache_processed_layers")
temp_chunk_dir = os.path.join(abs_save_dir, "temp_chunks")

# Clean temp dir for fresh run
if os.path.exists(temp_chunk_dir):
    shutil.rmtree(temp_chunk_dir)
os.makedirs(cache_dir, exist_ok=True)
os.makedirs(temp_chunk_dir, exist_ok=True)

def get_cache_path(layer_idx):
    return os.path.join(cache_dir, f"layer_{layer_idx}_data.pkl")

# Check which layers need processing
missing_layers = [l for l in config.candidate_layers_range if not os.path.exists(get_cache_path(l))]

# ---------------------------------------------------------
# 2. MULTI-LAYER HOOK (OPTIMIZED)
# ---------------------------------------------------------
class MultiLayerCaptureHook:
    """
    Captures last-token residual stream activations from multiple layers.

    Safety features:
    - Immediate CPU transfer to prevent VRAM spikes
    - Explicit clearing between batches
    - No double-capture protection (unnecessary with proper usage)
    """
    def __init__(self, target_layers):
        self.target_layers = set(target_layers)
        self.activations = {}
        self.handles = []

    def _make_hook(self, layer_idx):
        """Creates closure-based hook for specific layer."""
        def hook(module, input, output):
            # Handle tuple output (hidden_states, past_key_values)
            hs = output[0] if isinstance(output, tuple) else output
            # Extract last token + move to CPU immediately
            self.activations[layer_idx] = hs[:, -1, :].detach().cpu()
        return hook

    def register(self, model):
        """Attaches hooks to target layers."""
        if not hasattr(model, "model") or not hasattr(model.model, "layers"):
            raise ValueError("❌ Model architecture mismatch: Cannot find 'model.model.layers'")

        for idx, layer_module in enumerate(model.model.layers):
            if idx in self.target_layers:
                handle = layer_module.register_forward_hook(self._make_hook(idx))
                self.handles.append(handle)

        print(f"   ✅ Registered hooks on {len(self.handles)} layers")

    def clear(self):
        """Clears captured activations (call after each batch)."""
        self.activations = {}

    def remove(self):
        """Removes all hooks (call at end of extraction)."""
        for h in self.handles:
            h.remove()
        self.handles = []

# ---------------------------------------------------------
# 3. MAIN EXTRACTION LOGIC
# ---------------------------------------------------------
if len(missing_layers) == 0:
    print("✅ All layers already cached! Loading from disk...")
    all_layer_data = {}
    for l in tqdm(config.candidate_layers_range, desc="Loading Cache"):
        with open(get_cache_path(l), "rb") as f:
            all_layer_data[l] = pickle.load(f)
else:
    print(f"🚀 Processing Layers: {missing_layers}")

    # =====================================================
    # A. MODEL LOADING
    # =====================================================
    print("   ⬇️  Loading Mistral-7B (4-bit)...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

    tokenizer = AutoTokenizer.from_pretrained(config.base_model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    base_model = AutoModelForCausalLM.from_pretrained(
        config.base_model_name,
        quantization_config=bnb_config,
        device_map="auto",
        attn_implementation="sdpa",
        torch_dtype=torch.bfloat16
    )
    base_model.eval()
    device = base_model.device

    # =====================================================
    # B. PROMPT PREPROCESSING (WITH [INST] TAGS!)
    # =====================================================
    print("   📝 Pre-processing Prompts...")

    if 'group_id' not in bbq_merged_df.columns:
        raise ValueError("❌ 'group_id' missing. Run Grouping Step first!")

    # Token IDs for A/B/C (FIXED: use [-1] not [0])
    vocab_map = {
        'A': tokenizer.encode('A', add_special_tokens=False)[-1],
        'B': tokenizer.encode('B', add_special_tokens=False)[-1],
        'C': tokenizer.encode('C', add_special_tokens=False)[-1]
    }

    neutral_keywords = {
        "unknown", "can't", "cannot", "not be determined", "not known",
        "undetermined", "ambiguous", "not enough", "insufficient",
        "neither", "none", "don't know", "can't tell"
    }

    prompts = []
    metadata = []
    sys_msg = "Select the single best answer."

    for _, row in bbq_merged_df.iterrows():
        choices = [
            str(row.get('ans0', '')),
            str(row.get('ans1', '')),
            str(row.get('ans2', ''))
        ]

        # Find unknown option
        unknown_idx = -1
        for i, c in enumerate(choices):
            if any(kw in c.lower() for kw in neutral_keywords):
                unknown_idx = i
                break

        if row['context_condition'] == 'ambig' and unknown_idx == -1:
            continue

        # ✅ BUILD CONTENT (without "Answer:" yet)
        content = (
            f"{row['context']} {row['question']}\n"
            f"A. {choices[0]}\n"
            f"B. {choices[1]}\n"
            f"C. {choices[2]}"
        )

        # ✅ APPLY MISTRAL INSTRUCTION TEMPLATE
        inst_block = tokenizer.apply_chat_template(
            [{"role": "user", "content": f"{sys_msg}\n\n{content}"}],
            tokenize=False,
            add_generation_prompt=True
        )

        # ✅ APPEND TRIGGER OUTSIDE [INST] BLOCK
        full_prompt = inst_block + " Answer:"

        prompts.append(full_prompt)
        metadata.append({
            'target_loc': int(row['target_loc']),
            'unknown_idx': unknown_idx,
            'context_condition': row['context_condition'],
            'correct_label': int(row.get('label', -1)),
            'group_id': row['group_id']
        })

    print(f"   ✅ Valid Samples Ready: {len(prompts):,}")

    # =====================================================
    # 🔬 VERIFICATION: LAST TOKEN EXTRACTION CHECK
    # =====================================================
    print("\n" + "="*80)
    print(" 🔬 VERIFICATION: Last Token Extraction")
    print("="*80)

    # Take first 3 samples for verification
    verify_prompts = prompts[:3]
    verify_meta = metadata[:3]

    print(f"\n📝 Testing {len(verify_prompts)} sample prompts...")

    # Tokenize
    verify_inputs = tokenizer(
        verify_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=config.max_length,
        add_special_tokens=False
    ).to(device)

    # Create verification hook for one layer
    verify_layer = config.candidate_layers_range[0]

    class SingleLayerHook:
        def __init__(self):
            self.captured = None
        def __call__(self, module, input, output):
            hs = output[0] if isinstance(output, tuple) else output
            self.captured = hs[:, -1, :].detach().cpu()

    verify_hook = SingleLayerHook()
    verify_handle = base_model.model.layers[verify_layer].register_forward_hook(verify_hook)

    # Forward pass
    with torch.inference_mode():
        verify_outputs = base_model(**verify_inputs, output_hidden_states=True)

    # Remove hook
    verify_handle.remove()

    # Extract data
    input_ids = verify_inputs["input_ids"]
    logits = verify_outputs.logits
    hidden_from_hook = verify_hook.captured
    hidden_from_output = verify_outputs.hidden_states[verify_layer]

    print("\n" + "="*80)
    print("📊 EXTRACTION VERIFICATION TABLE")
    print("="*80)
    print(f"{'Sample':<8} | {'Last Token':<15} | {'Token ID':<10} | {'Hook Shape':<20} | {'Match?':<8}")
    print("-"*80)

    all_match = True

    for i in range(len(verify_prompts)):
        # Get last token info
        last_token_id = input_ids[i, -1].item()
        last_token_text = tokenizer.decode([last_token_id])

        # Get hidden states
        hook_hidden = hidden_from_hook[i]
        output_hidden = hidden_from_output[i, -1, :]

        # Check if they match
        match = torch.allclose(hook_hidden, output_hidden.cpu(), rtol=1e-4)
        all_match = all_match and match

        match_symbol = "✅" if match else "❌"

        print(f"{i:<8} | {repr(last_token_text):<15} | {last_token_id:<10} | {tuple(hook_hidden.shape):<20} | {match_symbol}")

    print("-"*80)

    # Verify dimensions
    print(f"\n🔍 DIMENSION CHECK:")
    print(f"   Logits shape:        {logits.shape} → [Batch, Seq, Vocab={logits.shape[-1]}]")
    print(f"   Hook hidden shape:   {hidden_from_hook.shape} → [Batch, Hidden={hidden_from_hook.shape[-1]}]")
    print(f"   Output hidden shape: {hidden_from_output.shape} → [Batch, Seq, Hidden={hidden_from_output.shape[-1]}]")

    print(f"\n🎯 WHAT WE EXTRACT:")
    print(f"   ✅ Residual Stream: hidden_states[:, -1, :] from layer {verify_layer}")
    print(f"   ✅ Dimension: {hidden_from_hook.shape[-1]} (model hidden size)")
    print(f"   ❌ NOT extracting: Logits (only used for labeling)")
    print(f"   ❌ NOT extracting: Full sequence (only last token)")

    print(f"\n🔬 HOOK ACCURACY:")
    if all_match:
        print(f"   ✅ Hook captures EXACT same values as output_hidden_states")
        print(f"   ✅ Extraction is CORRECT")
    else:
        print(f"   ❌ WARNING: Hook values don't match output_hidden_states!")

    # Check last token is colon
    print(f"\n✅ LAST TOKEN VERIFICATION:")
    for i in range(len(verify_prompts)):
        last_token_id = input_ids[i, -1].item()
        last_token_text = tokenizer.decode([last_token_id])
        is_colon = ":" in last_token_text
        symbol = "✅" if is_colon else "❌"
        print(f"   Sample {i}: '{last_token_text}' {symbol}")

    print("\n" + "="*80)
    print("✅ Verification Complete. Proceeding with full extraction...")
    print("="*80 + "\n")

    # Cleanup
    del verify_hook, verify_inputs, verify_outputs
    torch.cuda.empty_cache()

    # =====================================================
    # C. BATCHED EXTRACTION WITH LIVE MONITORING
    # =====================================================
    BATCH_SIZE = config.extraction_batch_size
    CHUNK_SAVE_FREQ = 2000
    DEBUG_LIMIT = 3

    # Buffers
    chunk_buffers = {l: [] for l in config.candidate_layers_range}
    label_buffer = []
    group_buffer = []
    stats = {'biased': 0, 'unbiased': 0, 'skipped': 0}
    chunk_counter = 0
    debug_counter = 0

    # 📊 Monitoring: Track extraction quality
    extraction_log = {
        'batch': [],
        'valid_samples': [],
        'biased_count': [],
        'unbiased_count': [],
        'biased_pct': [],
        'unbiased_pct': []
    }

    # Register hooks
    hook_manager = MultiLayerCaptureHook(config.candidate_layers_range)
    hook_manager.register(base_model)

    try:
        batch_num = 0
        for i in tqdm(range(0, len(prompts), BATCH_SIZE), desc="⚡ Extraction"):
            batch_prompts = prompts[i : i + BATCH_SIZE]
            batch_meta = metadata[i : i + BATCH_SIZE]

            # Tokenize (template already added BOS)
            inputs = tokenizer(
                batch_prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=config.max_length,
                add_special_tokens=False
            ).to(device)

            # Forward pass (hooks capture automatically)
            with torch.inference_mode():
                outputs = base_model(**inputs)

            # Get predictions
            logits = outputs.logits[:, -1, :]
            probs = torch.softmax(logits, dim=-1)

            valid_batch_indices = []
            batch_biased = 0
            batch_unbiased = 0

            for j, meta in enumerate(batch_meta):
                p_a = probs[j, vocab_map['A']].item()
                p_b = probs[j, vocab_map['B']].item()
                p_c = probs[j, vocab_map['C']].item()
                pred_idx = np.argmax([p_a, p_b, p_c])

                label = -1
                cond = meta['context_condition']
                tgt = meta['target_loc']
                unk = meta['unknown_idx']
                corr = meta['correct_label']

                # Labeling logic
                if cond == 'ambig':
                    if pred_idx == tgt:
                        label = config.LABEL_BIASED
                        stats['biased'] += 1
                        batch_biased += 1
                    elif pred_idx == unk:
                        label = config.LABEL_UNBIASED
                        stats['unbiased'] += 1
                        batch_unbiased += 1
                    else:
                        stats['skipped'] += 1

                elif cond == 'disambig':
                    if corr != -1:
                        if pred_idx == tgt and pred_idx != corr:
                            label = config.LABEL_BIASED
                            stats['biased'] += 1
                            batch_biased += 1
                        elif pred_idx == corr and corr != tgt:
                            label = config.LABEL_UNBIASED
                            stats['unbiased'] += 1
                            batch_unbiased += 1
                        else:
                            stats['skipped'] += 1
                    else:
                        stats['skipped'] += 1

                # Debug output
                if debug_counter < DEBUG_LIMIT and label != -1:
                    print(f"\n🔍 DEBUG #{debug_counter+1}:")
                    print(f"   Condition: {cond}")
                    print(f"   Prediction: {['A','B','C'][pred_idx]}")
                    print(f"   Probs: A={p_a:.3f}, B={p_b:.3f}, C={p_c:.3f}")
                    print(f"   Label: {label}")
                    print(f"   Extracted shape: {hook_manager.activations[config.candidate_layers_range[0]].shape}")
                    debug_counter += 1

                if label != -1:
                    label_buffer.append(label)
                    group_buffer.append(meta['group_id'])
                    valid_batch_indices.append(j)

            # 📊 Log batch stats
            if len(valid_batch_indices) > 0:
                extraction_log['batch'].append(batch_num)
                extraction_log['valid_samples'].append(len(valid_batch_indices))
                extraction_log['biased_count'].append(batch_biased)
                extraction_log['unbiased_count'].append(batch_unbiased)
                extraction_log['biased_pct'].append(batch_biased / len(valid_batch_indices) * 100)
                extraction_log['unbiased_pct'].append(batch_unbiased / len(valid_batch_indices) * 100)

            batch_num += 1

            # Retrieve activations from hooks
            if valid_batch_indices:
                for l in config.candidate_layers_range:
                    raw_acts = hook_manager.activations[l]
                    valid_acts = raw_acts[valid_batch_indices].float().numpy()
                    chunk_buffers[l].extend(valid_acts)

            # Clear hooks for next batch
            hook_manager.clear()

            # ✅ PROGRESSIVE SAVING (every 2000 samples)
            if len(label_buffer) >= CHUNK_SAVE_FREQ:
                print(f"\n💾 Saving chunk {chunk_counter} ({len(label_buffer)} samples)...")
                np.save(f"{temp_chunk_dir}/labels_{chunk_counter}.npy", np.array(label_buffer))
                np.save(f"{temp_chunk_dir}/groups_{chunk_counter}.npy", np.array(group_buffer))

                for l in config.candidate_layers_range:
                    np.save(f"{temp_chunk_dir}/layer_{l}_{chunk_counter}.npy", np.array(chunk_buffers[l]))
                    chunk_buffers[l] = []

                label_buffer = []
                group_buffer = []
                chunk_counter += 1
                gc.collect()

    finally:
        # ✅ CLEANUP: Remove hooks before model deletion
        print("\n🧹 Cleaning up hooks...")
        hook_manager.remove()
        del base_model
        torch.cuda.empty_cache()

    # =====================================================
    # 📊 PLOT EXTRACTION QUALITY
    # =====================================================
    print("\n" + "="*80)
    print(" 📊 EXTRACTION QUALITY MONITORING")
    print("="*80)

    if len(extraction_log['batch']) > 0:
        fig, axes = plt.subplots(2, 2, figsize=(16, 10))

        # Plot 1: Valid samples per batch
        axes[0, 0].plot(extraction_log['batch'], extraction_log['valid_samples'],
                       color='steelblue', linewidth=2, marker='o', markersize=3)
        axes[0, 0].set_title("Valid Samples per Batch", fontweight='bold', fontsize=14)
        axes[0, 0].set_xlabel("Batch Number")
        axes[0, 0].set_ylabel("Valid Samples")
        axes[0, 0].grid(True, alpha=0.3)

        # Plot 2: Label distribution (percentage)
        axes[0, 1].plot(extraction_log['batch'], extraction_log['biased_pct'],
                       label='Biased', color='#E74C3C', linewidth=2)
        axes[0, 1].plot(extraction_log['batch'], extraction_log['unbiased_pct'],
                       label='Unbiased', color='#27AE60', linewidth=2)
        axes[0, 1].set_title("Label Distribution per Batch (%)", fontweight='bold', fontsize=14)
        axes[0, 1].set_xlabel("Batch Number")
        axes[0, 1].set_ylabel("Percentage (%)")
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        axes[0, 1].set_ylim([0, 100])

        # Plot 3: Cumulative counts
        cumulative_biased = np.cumsum(extraction_log['biased_count'])
        cumulative_unbiased = np.cumsum(extraction_log['unbiased_count'])
        axes[1, 0].plot(extraction_log['batch'], cumulative_biased,
                       label='Biased', color='#E74C3C', linewidth=2)
        axes[1, 0].plot(extraction_log['batch'], cumulative_unbiased,
                       label='Unbiased', color='#27AE60', linewidth=2)
        axes[1, 0].set_title("Cumulative Label Counts", fontweight='bold', fontsize=14)
        axes[1, 0].set_xlabel("Batch Number")
        axes[1, 0].set_ylabel("Cumulative Count")
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)

        # Plot 4: Summary statistics
        axes[1, 1].axis('off')
        summary_text = f"""
        📊 EXTRACTION SUMMARY
        {'='*40}

        Total Batches:     {len(extraction_log['batch']):,}
        Total Valid:       {sum(extraction_log['valid_samples']):,}

        Biased Samples:    {stats['biased']:,} ({stats['biased']/(stats['biased']+stats['unbiased'])*100:.1f}%)
        Unbiased Samples:  {stats['unbiased']:,} ({stats['unbiased']/(stats['biased']+stats['unbiased'])*100:.1f}%)
        Skipped Samples:   {stats['skipped']:,}

        Avg Valid/Batch:   {np.mean(extraction_log['valid_samples']):.1f}
        Avg Biased %:      {np.mean(extraction_log['biased_pct']):.1f}%
        Avg Unbiased %:    {np.mean(extraction_log['unbiased_pct']):.1f}%

        Balance Ratio:     {stats['biased']/stats['unbiased']:.3f}
        """
        axes[1, 1].text(0.1, 0.5, summary_text, fontsize=12, family='monospace',
                       verticalalignment='center')

        plt.tight_layout()
        plot_path = os.path.join(abs_save_dir, "extraction_quality.png")
        plt.savefig(plot_path, dpi=150, bbox_inches='tight')
        plt.show()

        print(f"✅ Quality plot saved to: {plot_path}")

    # =====================================================
    # D. CONSOLIDATION & FINAL SAVING
    # =====================================================
    print(f"\n📊 Final Stats: Biased={stats['biased']:,} | Unbiased={stats['unbiased']:,} | Skipped={stats['skipped']:,}")
    print("\n🏭 Consolidating chunks and saving to cache...")

    # Save last chunk
    if len(label_buffer) > 0:
        print(f"💾 Saving final chunk {chunk_counter}...")
        np.save(f"{temp_chunk_dir}/labels_{chunk_counter}.npy", np.array(label_buffer))
        np.save(f"{temp_chunk_dir}/groups_{chunk_counter}.npy", np.array(group_buffer))
        for l in config.candidate_layers_range:
            np.save(f"{temp_chunk_dir}/layer_{l}_{chunk_counter}.npy", np.array(chunk_buffers[l]))

    # Merge all chunks
    lbl_list, grp_list = [], []
    for i in range(chunk_counter + 1):
        lbl_path = f"{temp_chunk_dir}/labels_{i}.npy"
        if os.path.exists(lbl_path):
            lbl_list.append(np.load(lbl_path))
            grp_list.append(np.load(f"{temp_chunk_dir}/groups_{i}.npy"))

    if not lbl_list:
        raise ValueError("❌ No valid samples extracted! Check prompt/labeling logic.")

    final_labels = np.concatenate(lbl_list)
    final_groups = np.concatenate(grp_list)
    all_layer_data = {}

    print(f"\n✅ Consolidated {len(final_labels):,} samples across {chunk_counter + 1} chunks")

    # Process each layer
    for l in tqdm(config.candidate_layers_range, desc="💾 Saving Layers"):
        print(f"\n   ⚙️ Layer {l}: Consolidating activations...")

        act_list = []
        for i in range(chunk_counter + 1):
            act_path = f"{temp_chunk_dir}/layer_{l}_{i}.npy"
            if os.path.exists(act_path):
                act_list.append(np.load(act_path))

        full_acts = np.concatenate(act_list)
        print(f"      Shape: {full_acts.shape}")

        # Run pipeline (split/balance/scale)
        X_train, y_train, X_val, y_val, scaler = prepare_data_pipeline(
            full_acts, final_labels, final_groups, config
        )

        payload = {
            'X_train': X_train,
            'y_train': y_train,
            'X_val': X_val,
            'y_val': y_val,
            'scaler': scaler
        }

        # ✅ SAVE TO GOOGLE DRIVE
        cache_path = get_cache_path(l)
        print(f"      💾 Saving to: {cache_path}")
        with open(cache_path, "wb") as f:
            pickle.dump(payload, f)

        # Keep in RAM for training
        all_layer_data[l] = payload

        del full_acts
        gc.collect()

    # Cleanup temp files
    shutil.rmtree(temp_chunk_dir)
    print("\n" + "="*80)
    print("✅ EXTRACTION COMPLETE - All layers cached to Google Drive")
    print("="*80)

# 12. BAD Classifer training and evalution engine

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import os
from torch.utils.data import DataLoader, TensorDataset
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)
from copy import deepcopy

# ============================================================================
# 0. REPRODUCIBILITY
# ============================================================================
def set_seed(seed=42):
    """Ensures reproducible results."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ============================================================================
# 1. VISUALIZATION SUITE
# ============================================================================
def plot_training_dashboard(history, save_path=None):
    """Generates an industry-standard training dashboard."""
    epochs = range(1, len(history['train_loss']) + 1)

    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))

    # Panel 1: Loss
    sns.lineplot(x=epochs, y=history['train_loss'], ax=axes[0], label='Train Loss', color='#E24A33', linewidth=2)
    axes[0].set_title("📉 Optimization (Loss)", fontweight='bold')
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("BCE Loss")

    # Panel 2: Generalization (Balanced Acc)
    sns.lineplot(x=epochs, y=history['val_bal_acc'], ax=axes[1], label='Val Balanced Acc', color='#348ABD', linewidth=2)
    axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random Baseline')
    axes[1].set_title("⚖️ Generalization (Balanced Accuracy)", fontweight='bold')
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylim(0.4, 1.0)
    axes[1].legend()

    # Panel 3: Separability (AUC)
    sns.lineplot(x=epochs, y=history['val_auc'], ax=axes[2], label='Val AUC', color='#988ED5', linewidth=2)
    axes[2].set_title("🎯 Separability (ROC AUC)", fontweight='bold')
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylim(0.4, 1.0)
    axes[2].legend()

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"📊 Training dashboard saved to: {save_path}")

    plt.show()

def plot_model_evaluation(y_true, y_probs, y_preds, save_path=None):
    """Detailed confusion matrix and ROC analysis."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False,
                xticklabels=['Bias (0)', 'Unbias (1)'],
                yticklabels=['Bias (0)', 'Unbias (1)'])
    axes[0].set_title("Confusion Matrix", fontweight='bold')
    axes[0].set_ylabel("True Label")
    axes[0].set_xlabel("Predicted Label")

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    roc_auc = auc(fpr, tpr)

    axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.3f}')
    axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
    axes[1].set_xlim([0.0, 1.0])
    axes[1].set_ylim([0.0, 1.05])
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve', fontweight='bold')
    axes[1].legend(loc="lower right")

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"📊 Evaluation plot saved to: {save_path}")

    plt.show()

# ============================================================================
# 2. CHECKPOINT MANAGER
# ============================================================================
class CheckpointManager:
    """Handles saving and loading checkpoints to disk."""

    def __init__(self, checkpoint_dir, layer_idx):
        self.checkpoint_dir = checkpoint_dir
        self.layer_idx = layer_idx
        os.makedirs(checkpoint_dir, exist_ok=True)

        self.best_path = os.path.join(checkpoint_dir, f"layer_{layer_idx}_best.pt")
        self.last_path = os.path.join(checkpoint_dir, f"layer_{layer_idx}_last.pt")

    def save_checkpoint(self, model, optimizer, scheduler, scaler, epoch, score, history, is_best=False):
        """Saves checkpoint to disk."""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'score': score,
            'history': history,
            'layer_idx': self.layer_idx
        }

        # Always save last checkpoint
        torch.save(checkpoint, self.last_path)

        # Save best checkpoint
        if is_best:
            torch.save(checkpoint, self.best_path)
            print(f"   💾 Best checkpoint saved: {self.best_path}")

    def load_checkpoint(self, model, optimizer=None, scheduler=None, scaler=None, load_best=True):
        """Loads checkpoint from disk."""
        path = self.best_path if load_best else self.last_path

        if not os.path.exists(path):
            print(f"⚠️ No checkpoint found at {path}")
            return None

        checkpoint = torch.load(path)
        model.load_state_dict(checkpoint['model_state_dict'])

        if optimizer:
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        if scheduler:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        if scaler:
            scaler.load_state_dict(checkpoint['scaler_state_dict'])

        print(f"✅ Loaded checkpoint from epoch {checkpoint['epoch']} (score: {checkpoint['score']:.4f})")
        return checkpoint

# ============================================================================
# 3. TRAINING ENGINE (PRODUCTION-READY)
# ============================================================================
def train_bad_classifier(
    X_train, y_train, X_val, y_val,
    config,
    layer_idx: int,
    checkpoint_dir: str = None,
    resume: bool = False
):
    """
    Trains the BAD Classifier with full production features.

    Features:
    - Mixed precision training (AMP)
    - Gradient clipping
    - Learning rate scheduling
    - Early stopping
    - Disk checkpointing
    - Reproducibility
    """
    print("="*80)
    print(f" 🚀 TRAINING LAYER {layer_idx}")
    print("="*80)

    # Set seed for reproducibility
    set_seed(config.random_seed)

    # 1. Hardware Setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = torch.cuda.is_available()

    # 2. Data Preparation
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32)

    # Sanity checks
    assert len(X_train_t) > 0 and len(X_val_t) > 0, "Empty datasets!"
    assert X_train.shape[1] == X_val.shape[1], "Feature dimension mismatch!"

    train_ds = TensorDataset(X_train_t, y_train_t)
    val_ds = TensorDataset(X_val_t, y_val_t)

    train_loader = DataLoader(
        train_ds,
        batch_size=config.batch_size,
        shuffle=True,
        pin_memory=True,
        num_workers=0  # Set to 0 for Colab stability
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=config.batch_size,
        shuffle=False,
        pin_memory=True,
        num_workers=0
    )

    # 3. Model Initialization
    try:
        model = BADClassifier(
            input_dim=X_train.shape[1],
            dropout_rate=config.dropout_rate
        ).to(device)
    except NameError:
        raise NameError("❌ BADClassifier not defined!")

    # 4. Loss Function (with class imbalance handling)
    num_pos = (y_train == 1).sum()
    num_neg = (y_train == 0).sum()

    if num_pos == 0 or num_neg == 0:
        raise ValueError(f"❌ Class imbalance critical: pos={num_pos}, neg={num_neg}")

    pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # 5. Optimizer & Scheduler
    optimizer = optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
        betas=(0.9, 0.999),  # Standard Adam betas
        eps=1e-8  # Numerical stability
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='max',  # Maximize validation accuracy
        factor=0.5,
        patience=3,
        min_lr=1e-7  # Prevent LR from going too low
    )

    scaler = GradScaler(enabled=use_amp)

    # 6. Checkpoint Manager
    if checkpoint_dir is None:
        checkpoint_dir = config.local_save_dir

    ckpt_manager = CheckpointManager(checkpoint_dir, layer_idx)

    # Resume from checkpoint if requested
    start_epoch = 0
    history = {'train_loss': [], 'val_bal_acc': [], 'val_auc': []}
    best_score = 0.0
    patience_counter = 0

    if resume:
        checkpoint = ckpt_manager.load_checkpoint(
            model, optimizer, scheduler, scaler, load_best=False
        )
        if checkpoint:
            start_epoch = checkpoint['epoch'] + 1
            history = checkpoint['history']
            best_score = checkpoint['score']
            print(f"🔄 Resuming from epoch {start_epoch}")

    # 7. Training Info
    print(f"   ⚙️  Device: {device} | AMP: {use_amp} | Pos Weight: {pos_weight.item():.3f}")
    print(f"   📊 Train: {len(X_train):,} | Val: {len(X_val):,} | Batch: {config.batch_size}")
    print(f"   🎯 Class Balance: Pos={num_pos:,} ({num_pos/(num_pos+num_neg)*100:.1f}%) | Neg={num_neg:,} ({num_neg/(num_pos+num_neg)*100:.1f}%)")
    print("-" * 85)
    print(f"   {'Epoch':<6} | {'Loss':<8} | {'Bal Acc':<8} | {'AUC':<8} | {'LR':<9} | {'Status'}")
    print("-" * 85)

    # 8. Training Loop
    for epoch in range(start_epoch, config.num_epochs):
        # ==================== TRAINING ====================
        model.train()
        train_loss = 0.0
        train_batches = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)  # More efficient than zero_grad()

            # Mixed precision forward pass
            with autocast(device_type='cuda', enabled=use_amp):
                logits = model(X_batch).squeeze(-1)
                loss = criterion(logits, y_batch)

            # Backward pass with gradient scaling
            scaler.scale(loss).backward()

            # Gradient clipping (unscale first!)
            if config.gradient_clip_norm > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    config.gradient_clip_norm
                )

            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            train_batches += 1

        avg_train_loss = train_loss / train_batches

        # ==================== VALIDATION ====================
        model.eval()
        val_probs = []
        val_labels = []

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(device, non_blocking=True)

                with autocast(device_type='cuda', enabled=use_amp):
                    logits = model(X_batch).squeeze(-1)
                    probs = torch.sigmoid(logits)

                val_probs.extend(probs.float().cpu().numpy())
                val_labels.extend(y_batch.cpu().numpy())

        # ==================== METRICS ====================
        val_probs = np.array(val_probs)
        val_labels = np.array(val_labels)
        val_preds = (val_probs >= 0.5).astype(int)

        val_bal_acc = balanced_accuracy_score(val_labels, val_preds)

        try:
            val_auc = roc_auc_score(val_labels, val_probs)
        except ValueError:
            val_auc = 0.5  # If only one class present

        # Update history
        history['train_loss'].append(avg_train_loss)
        history['val_bal_acc'].append(val_bal_acc)
        history['val_auc'].append(val_auc)

        # ==================== CHECKPOINTING ====================
        is_best = False
        status_msg = ""

        if val_bal_acc > best_score:
            best_score = val_bal_acc
            is_best = True
            patience_counter = 0
            status_msg = "⭐ Best"
        else:
            patience_counter += 1

        # Save checkpoint
        ckpt_manager.save_checkpoint(
            model, optimizer, scheduler, scaler,
            epoch, val_bal_acc, history, is_best=is_best
        )

        # ==================== LOGGING ====================
        current_lr = optimizer.param_groups[0]['lr']
        print(f"   {epoch+1:<6} | {avg_train_loss:<8.4f} | {val_bal_acc:<8.4f} | {val_auc:<8.4f} | {current_lr:<9.2e} | {status_msg}")

        # ==================== SCHEDULER ====================
        scheduler.step(val_bal_acc)

        # ==================== EARLY STOPPING ====================
        if patience_counter >= config.early_stopping_patience:
            print(f"\n🛑 Early stopping triggered at epoch {epoch+1}")
            break

    # ==================== LOAD BEST MODEL ====================
    print("\n" + "="*80)
    print(" 📊 LOADING BEST MODEL FOR EVALUATION")
    print("="*80)

    best_checkpoint = ckpt_manager.load_checkpoint(model, load_best=True)

    # ==================== FINAL EVALUATION ====================
    model.eval()
    final_probs = []
    final_labels = []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device, non_blocking=True)

            with autocast(device_type='cuda', enabled=use_amp):
                logits = model(X_batch).squeeze(-1)
                probs = torch.sigmoid(logits)

            final_probs.extend(probs.cpu().numpy())
            final_labels.extend(y_batch.cpu().numpy())

    final_probs = np.array(final_probs)
    final_labels = np.array(final_labels)
    final_preds = (final_probs >= 0.5).astype(int)

    # Classification Report
    print("\n📝 Final Classification Report:")
    print(classification_report(
        final_labels, final_preds,
        target_names=['Biased (0)', 'Unbiased (1)'],
        digits=4
    ))

    # Plots
    plot_path_dash = os.path.join(checkpoint_dir, f"layer_{layer_idx}_training.png")
    plot_path_eval = os.path.join(checkpoint_dir, f"layer_{layer_idx}_evaluation.png")

    plot_training_dashboard(history, save_path=plot_path_dash)
    plot_model_evaluation(final_labels, final_probs, final_preds, save_path=plot_path_eval)

    print("\n" + "="*80)
    print(f" ✅ TRAINING COMPLETE - Layer {layer_idx}")
    print(f" 🎯 Best Score: {best_score:.4f}")
    print("="*80 + "\n")

    return model, best_score, history

# ============================================================================
# 4. MULTI-LAYER TRAINING ORCHESTRATOR
# ============================================================================
def train_all_layers(all_layer_data, config):
    """
    Trains BAD classifier on all layers and selects the best.

    This is THE CRITICAL FUNCTION for finding the optimal layer!
    """
    print("\n" + "="*80)
    print(" 🏆 MULTI-LAYER TRAINING ORCHESTRATOR")
    print("="*80)

    layer_results = []

    for layer_idx in config.candidate_layers_range:
        print(f"\n{'='*80}")
        print(f" 🔷 LAYER {layer_idx}/{config.candidate_layers_range[-1]}")
        print(f"{'='*80}")

        # Get data for this layer
        if layer_idx not in all_layer_data:
            print(f"⚠️ Layer {layer_idx} data not found, skipping...")
            continue

        layer_data = all_layer_data[layer_idx]
        X_train = layer_data['X_train']
        y_train = layer_data['y_train']
        X_val = layer_data['X_val']
        y_val = layer_data['y_val']

        # Train this layer
        try:
            model, score, history = train_bad_classifier(
                X_train, y_train, X_val, y_val,
                config, layer_idx,
                checkpoint_dir=config.local_save_dir,
                resume=False
            )

            layer_results.append({
                'layer': layer_idx,
                'score': score,
                'model': model,
                'history': history
            })

        except Exception as e:
            print(f"❌ Layer {layer_idx} training failed: {e}")
            continue

    # ==================== SELECT BEST LAYER ====================
    if not layer_results:
        raise ValueError("❌ No layers were successfully trained!")

    # Sort by score (descending)
    layer_results_sorted = sorted(layer_results, key=lambda x: x['score'], reverse=True)
    best_result = layer_results_sorted[0]

    print("\n" + "="*80)
    print(" 🏆 BEST LAYER SELECTION")
    print("="*80)
    print(f"\n{'Layer':<8} | {'Score':<10} | {'Rank'}")
    print("-" * 40)

    for rank, result in enumerate(layer_results_sorted, 1):
        symbol = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉" if rank == 3 else "  "
        print(f"{result['layer']:<8} | {result['score']:<10.4f} | {symbol}")

    print("\n" + "="*80)
    print(f" 🎯 OPTIMAL LAYER: {best_result['layer']}")
    print(f" 🎯 BEST SCORE: {best_result['score']:.4f}")
    print("="*80 + "\n")

    return best_result['model'], best_result['layer'], best_result['score'], layer_results

print("✅ Training Engine Ready (Production Version)")

#13.

In [ ]:


import gc
import torch
import matplotlib.pyplot as plt
import seaborn as sns

print("="*80)
print(" 🧠 PHASE TRANSITION: EXTRACTION → TRAINING")
print("="*80 + "\n")

# ---------------------------------------------------------
# 1. AGGRESSIVE MEMORY CLEANUP (CRITICAL FOR COLAB!)
# ---------------------------------------------------------
# Google/OpenAI Best Practice: Explicit phase separation
# Why: LLM inference (4-8GB) vs Classifier training (1-2GB)
# Different VRAM profiles require explicit cleanup

print("🗑️  Unloading extraction phase artifacts...")

# Kill the LLM
if 'base_model' in globals():
    del base_model
    print("   ✅ Base model unloaded")

if 'tokenizer' in globals():
    del tokenizer
    print("   ✅ Tokenizer unloaded")

# Kill extraction buffers
if 'prompts' in globals():
    del prompts
if 'metadata' in globals():
    del metadata

# Aggressive garbage collection
gc.collect()
torch.cuda.empty_cache()

# GPU defragmentation (Google's internal practice)
if torch.cuda.is_available():
    try:
        torch.cuda.ipc_collect()  # Inter-process cleanup
    except:
        pass

    # Report available memory
    free_mem, total_mem = torch.cuda.mem_get_info()
    free_gb = free_mem / 1024**3
    total_gb = total_mem / 1024**3
    print(f"\n💾 GPU Memory: {free_gb:.2f}/{total_gb:.2f} GB free")

    if free_gb < 2.0:
        print("⚠️ WARNING: Low GPU memory. Consider restarting runtime.")

print("\n✅ Memory cleanup complete. Ready for training phase.")

# ---------------------------------------------------------
# 2. TRAIN ALL LAYERS (Using Production Engine)
# ---------------------------------------------------------
print("\n" + "="*80)
print(" 🚀 MULTI-LAYER BAD CLASSIFIER TRAINING")
print("="*80 + "\n")

# Verify data availability
if 'all_layer_data' not in globals() or not all_layer_data:
    raise ValueError("❌ 'all_layer_data' not found. Run extraction first!")

# Run multi-layer training orchestrator
try:
    best_model, best_layer, best_score, layer_results = train_all_layers(
        all_layer_data,
        config
    )
except Exception as e:
    print(f"❌ Training failed: {e}")
    raise

# ---------------------------------------------------------
# 3. SAVE BEST MODEL & SCALER
# ---------------------------------------------------------
print("\n" + "="*80)
print(" 💾 SAVING ARTIFACTS")
print("="*80)

# Save best model
model_path = os.path.join(config.local_save_dir, f"bad_classifier_layer_{best_layer}.pt")
torch.save({
    'model_state_dict': best_model.state_dict(),
    'layer_idx': best_layer,
    'score': best_score,
    'config': config
}, model_path)
print(f"✅ Best model saved: {model_path}")

# Save scaler
scaler_path = os.path.join(config.local_save_dir, f"bad_scaler_layer_{best_layer}.pkl")
import pickle
with open(scaler_path, 'wb') as f:
    pickle.dump(all_layer_data[best_layer]['scaler'], f)
print(f"✅ Scaler saved: {scaler_path}")

# ---------------------------------------------------------
# 4. COMPARATIVE VISUALIZATION
# ---------------------------------------------------------
print("\n" + "="*80)
print(" 📊 LAYER COMPARISON ANALYSIS")
print("="*80 + "\n")

# Extract layer indices and scores
layers = [r['layer'] for r in layer_results]
scores = [r['score'] for r in layer_results]

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Layer performance curve
axes[0].plot(layers, scores, marker='o', linewidth=2.5, markersize=8, color='#6c5ce7')
axes[0].axvline(x=best_layer, color='red', linestyle='--', linewidth=2, label=f'Best: Layer {best_layer}')
axes[0].axhline(y=0.5, color='gray', linestyle=':', alpha=0.5, label='Random Baseline')
axes[0].set_title(f"BAD Classifier Performance by Layer\n({config.base_model_name})",
                  fontweight='bold', fontsize=14)
axes[0].set_xlabel("Layer Index", fontsize=12)
axes[0].set_ylabel("Balanced Accuracy", fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0.45, 1.0])

# Plot 2: Top 5 layers bar chart
sorted_results = sorted(layer_results, key=lambda x: x['score'], reverse=True)[:5]
top_layers = [r['layer'] for r in sorted_results]
top_scores = [r['score'] for r in sorted_results]
colors = ['#e74c3c' if l == best_layer else '#3498db' for l in top_layers]

axes[1].barh(range(len(top_layers)), top_scores, color=colors)
axes[1].set_yticks(range(len(top_layers)))
axes[1].set_yticklabels([f"Layer {l}" for l in top_layers])
axes[1].set_xlabel("Balanced Accuracy", fontsize=12)
axes[1].set_title("Top 5 Performing Layers", fontweight='bold', fontsize=14)
axes[1].axvline(x=0.5, color='gray', linestyle=':', alpha=0.5)
axes[1].set_xlim([0.45, 1.0])

# Add score labels
for i, (layer, score) in enumerate(zip(top_layers, top_scores)):
    axes[1].text(score + 0.01, i, f"{score:.4f}",
                va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plot_path = os.path.join(config.local_save_dir, "layer_comparison.png")
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Comparison plot saved: {plot_path}")

# ---------------------------------------------------------
# 5. SUMMARY REPORT
# ---------------------------------------------------------
print("\n" + "="*80)
print(" 📋 TRAINING SUMMARY")
print("="*80)
print(f"""
Model:           {config.base_model_name}
Layers Trained:  {len(layer_results)}
Best Layer:      {best_layer}
Best Score:      {best_score:.4f}
Training Time:   See individual layer logs above
Checkpoint Dir:  {config.local_save_dir}

Next Steps:
1. Use best_model and best_layer for DSV computation
2. Run evaluation notebook to verify performance
3. Generate publication plots
""")

# ---------------------------------------------------------
# 6. EXPORT VARIABLES FOR NEXT CELLS
# ---------------------------------------------------------
# Make these available for subsequent cells
bad_classifier = best_model
bad_scaler = all_layer_data[best_layer]['scaler']
optimal_layer = best_layer

print(f"✅ Variables exported: bad_classifier, bad_scaler, optimal_layer")
print("="*80 + "\n")


# 14: BEST LAYER TRAINING DYNAMICS VISUALIZATION


In [ ]:


import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("="*80)
print(f" 📊 TRAINING DYNAMICS ANALYSIS - LAYER {best_layer}")
print("="*80 + "\n")

# ---------------------------------------------------------
# 1. EXTRACT HISTORY FOR BEST LAYER
# ---------------------------------------------------------
# New structure: layer_results is a list of dicts
# Find the result for best_layer
best_result = next((r for r in layer_results if r['layer'] == best_layer), None)

if best_result is None:
    raise ValueError(f"❌ No results found for layer {best_layer}")

history = best_result['history']
epochs = range(1, len(history['train_loss']) + 1)

print(f"📈 Visualizing {len(epochs)} epochs of training data...")

# ---------------------------------------------------------
# 2. CREATE DASHBOARD
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    f"Training Dynamics - Layer {best_layer} (Optimal Model)",
    fontsize=16,
    fontweight='bold'
)

# =========================================================
# PANEL 1: OPTIMIZATION (LOSS CONVERGENCE)
# =========================================================
loss_data = history['train_loss']

# Plot raw loss
axes[0].plot(epochs, loss_data, 'o-', alpha=0.3, color='#3498db', label='Training Loss')

# Add smoothed trend (if enough epochs)
if len(loss_data) > 5:
    window = min(5, len(loss_data) // 2)  # Adaptive window size
    smoothed = np.convolve(loss_data, np.ones(window)/window, mode='valid')
    smooth_epochs = range(window//2 + 1, len(smoothed) + window//2 + 1)
    axes[0].plot(smooth_epochs, smoothed, '-', linewidth=3, color='#2980b9', label='Smoothed Trend')

# Styling
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('BCE Loss', fontsize=12)
axes[0].set_title('📉 Optimization Convergence', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='best', fontsize=10)

# Add annotation for final loss
final_loss = loss_data[-1]
axes[0].annotate(
    f'Final: {final_loss:.4f}',
    xy=(len(epochs), final_loss),
    xytext=(len(epochs) * 0.7, final_loss * 1.1),
    arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
    fontsize=10,
    color='red',
    fontweight='bold'
)

# =========================================================
# PANEL 2: GENERALIZATION (VALIDATION METRICS)
# =========================================================
# Plot validation metrics
axes[1].plot(
    epochs, history['val_bal_acc'],
    'o-', linewidth=2.5, markersize=5,
    color='#27ae60', label='Balanced Acc'
)
axes[1].plot(
    epochs, history['val_auc'],
    's--', linewidth=2, markersize=5,
    color='#8e44ad', label='ROC AUC'
)

# Reference lines
axes[1].axhline(
    y=0.5, color='gray', linestyle=':',
    alpha=0.5, linewidth=1.5, label='Random (0.5)'
)
axes[1].axhline(
    y=best_score, color='#e74c3c', linestyle='--',
    alpha=0.7, linewidth=2, label=f'Best: {best_score:.4f}'
)

# Highlight best epoch
best_epoch_idx = np.argmax(history['val_bal_acc'])
best_epoch = best_epoch_idx + 1
best_acc = history['val_bal_acc'][best_epoch_idx]

axes[1].scatter(
    [best_epoch], [best_acc],
    s=200, c='red', marker='*',
    zorder=5, label='Best Model',
    edgecolors='darkred', linewidths=2
)

# Styling
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('⚖️ Validation Performance', fontsize=14, fontweight='bold')
axes[1].set_ylim([0.4, 1.0])
axes[1].legend(loc='lower right', fontsize=10)
axes[1].grid(True, alpha=0.3)

# Add annotation for best epoch
axes[1].annotate(
    f'Epoch {best_epoch}\nAcc: {best_acc:.4f}',
    xy=(best_epoch, best_acc),
    xytext=(best_epoch + len(epochs) * 0.15, best_acc - 0.05),
    arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
    fontsize=9,
    color='red',
    fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.8)
)

plt.tight_layout()

# ---------------------------------------------------------
# 3. SAVE FIGURE
# ---------------------------------------------------------
save_path = os.path.join(config.local_save_dir, f"layer_{best_layer}_training_dynamics.png")
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Training dynamics plot saved: {save_path}")

# ---------------------------------------------------------
# 4. DETAILED TRAINING SUMMARY
# ---------------------------------------------------------
print("\n" + "="*80)
print(" 📋 TRAINING SUMMARY - BEST MODEL")
print("="*80)

# Calculate statistics
final_train_loss = history['train_loss'][-1]
final_val_acc = history['val_bal_acc'][-1]
final_val_auc = history['val_auc'][-1]
best_val_acc = max(history['val_bal_acc'])
best_val_auc = max(history['val_auc'])

# Check for overfitting
loss_decrease = ((history['train_loss'][0] - final_train_loss) / history['train_loss'][0]) * 100

print(f"""
Layer:              {best_layer}
Total Epochs:       {len(epochs)}
Best Epoch:         {best_epoch}

Training Loss:
  - Initial:        {history['train_loss'][0]:.4f}
  - Final:          {final_train_loss:.4f}
  - Decrease:       {loss_decrease:.1f}%

Validation Metrics:
  - Best Bal Acc:   {best_val_acc:.4f} (Epoch {best_epoch})
  - Final Bal Acc:  {final_val_acc:.4f}
  - Best AUC:       {best_val_auc:.4f}
  - Final AUC:      {final_val_auc:.4f}

Model Status:       {'✅ Converged' if loss_decrease > 20 else '⚠️ May need more epochs'}
""")

# Check for overfitting
acc_drop = best_val_acc - final_val_acc
if acc_drop > 0.02:
    print(f"⚠️ WARNING: Accuracy dropped {acc_drop:.4f} from best to final epoch")
    print("   This may indicate overfitting or need for better early stopping.")
else:
    print("✅ No significant overfitting detected")

print("="*80 + "\n")

# 15 PUBLICATION-QUALITY LAYER SENSITIVITY PLOT (OPTIONAL)

In [ ]:


import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

print("\n" + "="*80)
print(" 📈 GENERATING PUBLICATION-READY LAYER SENSITIVITY PLOT")
print("="*80 + "\n")

# ---------------------------------------------------------
# 1. SAFETY CHECK
# ---------------------------------------------------------
if 'layer_results' not in globals() or not layer_results:
    raise ValueError("❌ 'layer_results' not found. Run Cell 11 (training) first!")

if 'best_layer' not in globals() or best_layer is None:
    raise ValueError("❌ 'best_layer' not found. Run Cell 11 (training) first!")

# ---------------------------------------------------------
# 2. EXTRACT DATA
# ---------------------------------------------------------
# New structure: layer_results is a list of dicts
sorted_layers = sorted([r['layer'] for r in layer_results])
accuracies = [next(r['score'] for r in layer_results if r['layer'] == l) for l in sorted_layers]

print(f"📊 Plotting {len(sorted_layers)} layers: {sorted_layers[0]} → {sorted_layers[-1]}")

# ---------------------------------------------------------
# 3. CREATE PUBLICATION-QUALITY FIGURE
# ---------------------------------------------------------
# Set publication style
sns.set_style("whitegrid")
sns.set_context("paper", font_scale=1.3)

fig, ax = plt.subplots(figsize=(10, 6), dpi=300)  # High DPI for papers

# Main line plot
ax.plot(
    sorted_layers,
    accuracies,
    marker='o',
    linewidth=2.5,
    markersize=8,
    color='#2c3e50',
    label='Balanced Accuracy',
    zorder=2
)

# Highlight best layer
best_score_val = next(r['score'] for r in layer_results if r['layer'] == best_layer)

# Red star for winner
ax.plot(
    best_layer,
    best_score_val,
    marker='*',
    markersize=25,
    color='#e74c3c',
    label=f'Optimal Layer {best_layer}',
    zorder=3,
    markeredgecolor='darkred',
    markeredgewidth=1.5
)

# Annotate best layer
ax.annotate(
    f"{best_score_val:.1%}",
    xy=(best_layer, best_score_val),
    xytext=(0, 20),
    textcoords="offset points",
    ha='center',
    fontsize=12,
    fontweight='bold',
    color='#e74c3c',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='#e74c3c', linewidth=2),
    arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0', color='#e74c3c', lw=1.5)
)

# Reference lines
ax.axhline(
    y=0.5,
    color='gray',
    linestyle=':',
    linewidth=1.5,
    label='Random Baseline',
    alpha=0.7,
    zorder=1
)

# Add shaded region for "good performance"
ax.axhspan(0.7, 1.0, alpha=0.1, color='green', zorder=0)
ax.text(
    sorted_layers[0] + 0.5, 0.85,
    'Strong Performance',
    fontsize=10,
    color='green',
    alpha=0.5,
    style='italic'
)

# ---------------------------------------------------------
# 4. STYLING
# ---------------------------------------------------------
# Get clean model name
model_name_short = config.base_model_name.split("/")[-1]

ax.set_title(
    f"Layer-wise Bias Detection Performance\n{model_name_short}",
    fontsize=16,
    fontweight='bold',
    pad=20
)
ax.set_xlabel("Layer Index", fontsize=14, fontweight='bold')
ax.set_ylabel("Balanced Accuracy", fontsize=14, fontweight='bold')

# Set axis limits
ax.set_ylim([0.45, 1.0])
ax.set_xlim([sorted_layers[0] - 0.5, sorted_layers[-1] + 0.5])

# Set x-ticks to show all layers
ax.set_xticks(sorted_layers)

# Legend
ax.legend(
    loc='lower right',
    frameon=True,
    framealpha=0.95,
    shadow=True,
    fontsize=11,
    edgecolor='black',
    fancybox=True
)

# Grid
ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

# Tight layout
plt.tight_layout()

# ---------------------------------------------------------
# 5. SAVE FIGURE
# ---------------------------------------------------------
os.makedirs(config.local_save_dir, exist_ok=True)
save_path = os.path.join(config.local_save_dir, "publication_layer_sensitivity.png")

try:
    plt.savefig(save_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"✅ Publication figure saved: {save_path}")

    # Also save as PDF for LaTeX papers
    pdf_path = save_path.replace('.png', '.pdf')
    plt.savefig(pdf_path, format='pdf', bbox_inches='tight', facecolor='white')
    print(f"✅ PDF version saved: {pdf_path}")

except Exception as e:
    print(f"⚠️ Could not save plot: {e}")

plt.show()

# ---------------------------------------------------------
# 6. STATISTICS SUMMARY
# ---------------------------------------------------------
print("\n" + "="*80)
print(" 📊 LAYER SENSITIVITY STATISTICS")
print("="*80)

# Calculate statistics
mean_acc = np.mean(accuracies)
std_acc = np.std(accuracies)
min_acc = min(accuracies)
max_acc = max(accuracies)
range_acc = max_acc - min_acc

worst_layer = sorted_layers[np.argmin(accuracies)]
second_best_layer = sorted_layers[np.argsort(accuracies)[-2]]

print(f"""
Layer Range:        {sorted_layers[0]} → {sorted_layers[-1]} ({len(sorted_layers)} layers)

Performance:
  - Best Layer:     {best_layer} (Acc: {best_score_val:.4f})
  - 2nd Best:       {second_best_layer} (Acc: {accuracies[sorted_layers.index(second_best_layer)]:.4f})
  - Worst Layer:    {worst_layer} (Acc: {min_acc:.4f})

Statistics:
  - Mean:           {mean_acc:.4f}
  - Std Dev:        {std_acc:.4f}
  - Range:          {range_acc:.4f}
  - Min:            {min_acc:.4f}
  - Max:            {max_acc:.4f}

Analysis:
  - Sensitivity:    {'High' if std_acc > 0.05 else 'Low'} (std={std_acc:.4f})
  - Best vs Mean:   {(best_score_val - mean_acc):.4f} better
  - Best vs Worst:  {(best_score_val - min_acc):.4f} better
""")

# Provide interpretation
if range_acc < 0.05:
    print("✅ Low sensitivity: Most layers perform similarly well")
elif range_acc < 0.10:
    print("⚖️ Moderate sensitivity: Some layers clearly better than others")
else:
    print("⚠️ High sensitivity: Layer selection is critical!")

print("="*80 + "\n")

# 16 Saving artifacts

In [ ]:

import os
import json
import pickle
import torch
from datetime import datetime
from safetensors.torch import save_file

print("="*80)
print(" 💾 SAVING ARTIFACTS (PRODUCTION READY)")
print("="*80 + "\n")

# ---------------------------------------------------------
# 1. SAFETY CHECKS
# ---------------------------------------------------------
required_vars = {
    'bad_classifier': 'Trained BAD classifier model',
    'bad_scaler': 'Data preprocessing scaler',
    'best_layer': 'Optimal layer index',
    'best_score': 'Best validation score'
}

missing_vars = []
for var_name, description in required_vars.items():
    if var_name not in globals() or globals()[var_name] is None:
        missing_vars.append(f"{var_name} ({description})")

if missing_vars:
    print("❌ CRITICAL: Missing required variables:")
    for var in missing_vars:
        print(f"   • {var}")
    print("\n⚠️ Please run Cell 11 (training) first!")
    raise ValueError("Cannot save artifacts without trained model")

# ---------------------------------------------------------
# 2. SETUP PATHS
# ---------------------------------------------------------
save_dir = config.local_save_dir
os.makedirs(save_dir, exist_ok=True)

print(f"📂 Save directory: {save_dir}\n")

try:
    # ---------------------------------------------------------
    # 3. PREPARE MODEL (ENSURE CPU)
    # ---------------------------------------------------------
    # Best practice: Always save models on CPU for portability
    model_to_save = bad_classifier.cpu()
    print("   ✅ Model moved to CPU for saving")

    # Extract model architecture info
    input_dim = model_to_save.linear.in_features

    print(f"\n📊 Model Summary:")
    print(f"   • Layer:      {best_layer}")
    print(f"   • Input Dim:  {input_dim}")
    print(f"   • Accuracy:   {best_score:.4f}")
    print(f"   • Timestamp:  {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

    # ---------------------------------------------------------
    # 4. SAVE MODEL WEIGHTS (DUAL FORMAT)
    # ---------------------------------------------------------
    print(f"\n💾 Saving model weights...")

    # A. SafeTensors (Modern, Fast, Secure)
    st_path = os.path.join(save_dir, "bad_classifier.safetensors")
    save_file(model_to_save.state_dict(), st_path)
    file_size_st = os.path.getsize(st_path) / 1024  # KB
    print(f"   ✅ SafeTensors:  bad_classifier.safetensors ({file_size_st:.1f} KB)")

    # B. PyTorch Binary (Legacy/HuggingFace Compatible)
    pt_path = os.path.join(save_dir, "bad_classifier.pt")
    torch.save(model_to_save.state_dict(), pt_path)
    file_size_pt = os.path.getsize(pt_path) / 1024  # KB
    print(f"   ✅ PyTorch:      bad_classifier.pt ({file_size_pt:.1f} KB)")

    # ---------------------------------------------------------
    # 5. SAVE COMPLETE MODEL (WITH ARCHITECTURE)
    # ---------------------------------------------------------
    # This saves the entire model object, not just weights
    # Useful for quick inference without reinstantiating
    full_model_path = os.path.join(save_dir, "bad_classifier_full.pt")
    torch.save({
        'model': model_to_save,
        'state_dict': model_to_save.state_dict(),
        'layer': best_layer,
        'score': best_score
    }, full_model_path)
    file_size_full = os.path.getsize(full_model_path) / 1024  # KB
    print(f"   ✅ Full Model:   bad_classifier_full.pt ({file_size_full:.1f} KB)")

    # ---------------------------------------------------------
    # 6. SAVE SCALER (CRITICAL FOR INFERENCE)
    # ---------------------------------------------------------
    print(f"\n💾 Saving data scaler...")
    scaler_path = os.path.join(save_dir, "bad_scaler.pkl")
    with open(scaler_path, "wb") as f:
        pickle.dump(bad_scaler, f)
    file_size_scaler = os.path.getsize(scaler_path) / 1024  # KB
    print(f"   ✅ Scaler:       bad_scaler.pkl ({file_size_scaler:.1f} KB)")

    # ---------------------------------------------------------
    # 7. SAVE METADATA & CONFIG
    # ---------------------------------------------------------
    print(f"\n💾 Saving configuration...")

    # Cast to native Python types for JSON serialization
    config_data = {
        'model_info': {
            'base_model': config.base_model_name,
            'layer_idx': int(best_layer),
            'input_dim': int(input_dim),
            'dropout_rate': float(config.dropout_rate),
            'architecture': 'LinearProbe_Dropout_Linear_Sigmoid'
        },
        'performance': {
            'metric_name': 'balanced_accuracy',
            'metric_value': float(best_score),
            'threshold': 0.5  # Classification threshold
        },
        'training': {
            'timestamp': datetime.now().isoformat(),
            'num_epochs': int(config.num_epochs),
            'batch_size': int(config.batch_size),
            'learning_rate': float(config.learning_rate),
            'early_stopping_patience': int(config.early_stopping_patience)
        },
        'inference': {
            'required_files': [
                'bad_classifier.pt or bad_classifier.safetensors',
                'bad_scaler.pkl',
                'config.json'
            ],
            'loading_example': 'See README.md for inference code'
        }
    }

    config_path = os.path.join(save_dir, "config.json")
    with open(config_path, 'w') as f:
        json.dump(config_data, f, indent=2)
    print(f"   ✅ Config:       config.json")

    # ---------------------------------------------------------
    # 8. SAVE LAYER COMPARISON RESULTS (OPTIONAL)
    # ---------------------------------------------------------
    if 'layer_results' in globals() and layer_results:
        print(f"\n💾 Saving layer comparison results...")

        layer_comparison = {
            'layers': [r['layer'] for r in layer_results],
            'scores': [float(r['score']) for r in layer_results],
            'best_layer': int(best_layer),
            'best_score': float(best_score)
        }

        comparison_path = os.path.join(save_dir, "layer_comparison.json")
        with open(comparison_path, 'w') as f:
            json.dump(layer_comparison, f, indent=2)
        print(f"   ✅ Comparison:   layer_comparison.json")

    # ---------------------------------------------------------
    # 9. CREATE INFERENCE README
    # ---------------------------------------------------------
    readme_content = f"""# BAD Classifier - Inference Guide

## Model Information
- Base Model: {config.base_model_name}
- Optimal Layer: {best_layer}
- Validation Accuracy: {best_score:.4f}
- Training Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Quick Start

### Load Model
```python
import torch
import pickle
import json

# Load config
with open('config.json', 'r') as f:
    config = json.load(f)

# Load model
model = torch.load('bad_classifier_full.pt')['model']
model.eval()

# Load scaler
with open('bad_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# OR load just weights
from your_model_file import BADClassifier
model = BADClassifier(input_dim=config['model_info']['input_dim'])
model.load_state_dict(torch.load('bad_classifier.pt'))
model.eval()
```

### Run Inference
```python
# Preprocess activation
activation_scaled = scaler.transform(activation.reshape(1, -1))

# Predict
with torch.no_grad():
    logit = model(torch.tensor(activation_scaled, dtype=torch.float32))
    prob = torch.sigmoid(logit).item()
    is_biased = prob < 0.5  # 0=biased, 1=unbiased

print(f"Bias probability: {{prob:.4f}}")
```

## Files
- `bad_classifier.pt` - Model weights (PyTorch)
- `bad_classifier.safetensors` - Model weights (SafeTensors)
- `bad_classifier_full.pt` - Complete model with architecture
- `bad_scaler.pkl` - Data preprocessing scaler
- `config.json` - Model metadata and hyperparameters
- `layer_comparison.json` - Performance across all layers (optional)

## Citation
If you use this model, please cite:
[Your paper details here]
"""

    readme_path = os.path.join(save_dir, "README.md")
    with open(readme_path, 'w') as f:
        f.write(readme_content)
    print(f"   ✅ README:       README.md")

    # ---------------------------------------------------------
    # 10. FINAL SUMMARY
    # ---------------------------------------------------------
    print("\n" + "="*80)
    print(" ✅ ARTIFACTS SAVED SUCCESSFULLY")
    print("="*80)
    print(f"\n📂 Location: {save_dir}\n")
    print("📦 Files created:")
    print("   • bad_classifier.safetensors  (Modern format)")
    print("   • bad_classifier.pt           (PyTorch format)")
    print("   • bad_classifier_full.pt      (Complete model)")
    print("   • bad_scaler.pkl              (Data preprocessor)")
    print("   • config.json                 (Model metadata)")
    print("   • layer_comparison.json       (Layer analysis)")
    print("   • README.md                   (Usage guide)")

    # Calculate total size
    total_size = sum([
        os.path.getsize(st_path),
        os.path.getsize(pt_path),
        os.path.getsize(full_model_path),
        os.path.getsize(scaler_path),
        os.path.getsize(config_path)
    ]) / (1024 * 1024)  # MB

    print(f"\n💾 Total size: {total_size:.2f} MB")
    print("\n🎯 Ready for deployment and DSV computation!")
    print("="*80 + "\n")

except Exception as e:
    print(f"\n❌ ERROR saving artifacts: {e}")
    import traceback
    traceback.print_exc()
    raise

# 17 Deploy to HuggingFace

In [ ]:
# ==========================================
# CELL 15: HUGGINGFACE DEPLOYMENT (PRODUCTION)
# ==========================================

import os
import shutil
import tempfile
import json
import torch
from huggingface_hub import HfApi, create_repo, login
from safetensors.torch import save_file
from datetime import datetime

print("="*80)
print(" 🚀 HUGGINGFACE DEPLOYMENT (BEST PRACTICES)")
print("="*80 + "\n")

# ---------------------------------------------------------
# 1. AUTHENTICATION
# ---------------------------------------------------------
REPO_ID = config.hf_repo_name

# Try multiple token sources
hf_token = os.environ.get('HF_TOKEN', None)

if not hf_token:
    try:
        from huggingface_hub import get_token
        hf_token = get_token()
    except:
        pass

if not hf_token:
    from getpass import getpass
    print("⚠️ HF_TOKEN not found in environment or CLI cache.")
    hf_token = getpass("Enter HuggingFace Token: ")

try:
    login(token=hf_token, add_to_git_credential=True)
    print("✅ Authenticated with HuggingFace\n")
except Exception as e:
    raise ValueError(f"❌ Authentication Failed: {e}")

# ---------------------------------------------------------
# 2. LOCATE SOURCE DIRECTORY
# ---------------------------------------------------------
print("📂 Locating artifacts...")

SOURCE_DIR = os.path.abspath(config.local_save_dir)

if not os.path.exists(SOURCE_DIR):
    raise FileNotFoundError(f"❌ Source directory not found: {SOURCE_DIR}")

print(f"✅ Source: {SOURCE_DIR}\n")

# ---------------------------------------------------------
# 3. LOAD METADATA (WITH NESTED STRUCTURE)
# ---------------------------------------------------------
config_path = os.path.join(SOURCE_DIR, "config.json")
if not os.path.exists(config_path):
    raise FileNotFoundError(f"❌ config.json not found in {SOURCE_DIR}")

with open(config_path, 'r') as f:
    saved_config = json.load(f)

# Extract from nested structure (matching Cell 14 format)
model_info = saved_config.get('model_info', {})
performance = saved_config.get('performance', {})

meta_layer = model_info.get('layer_idx', 'Unknown')
meta_score = performance.get('metric_value', 0.0)
meta_arch = model_info.get('architecture', 'Linear Probe')
base_model_name = model_info.get('base_model', config.base_model_name)
input_dim = model_info.get('input_dim', 4096)
dropout_rate = model_info.get('dropout_rate', 0.1)

print(f"📊 Model Metadata:")
print(f"   • Base Model: {base_model_name}")
print(f"   • Layer: {meta_layer}")
print(f"   • Accuracy: {meta_score:.4f}")
print(f"   • Architecture: {meta_arch}")
print()

# ---------------------------------------------------------
# 4. PREPARE UPLOAD DIRECTORY
# ---------------------------------------------------------
print("📦 Staging files for upload...")
hf_upload_dir = tempfile.mkdtemp(prefix="bad_hf_deploy_")

# ---------------------------------------------------------
# 5. COPY ESSENTIAL FILES ONLY
# ---------------------------------------------------------
# ✅ ONLY upload files needed for inference
essential_files = {
    'config.json': 'Model configuration',
    'bad_scaler.pkl': 'Data preprocessor',
    'README.md': 'Model card (will be generated)'
}

upload_manifest = []

# Copy config and scaler
for filename, description in essential_files.items():
    if filename == 'README.md':
        continue  # Will generate this

    src = os.path.join(SOURCE_DIR, filename)
    dst = os.path.join(hf_upload_dir, filename)

    if os.path.exists(src):
        shutil.copy(src, dst)
        upload_manifest.append(filename)
        print(f"   ✅ {filename} ({description})")
    else:
        raise FileNotFoundError(f"❌ Required file missing: {filename}")

# ---------------------------------------------------------
# 6. HANDLE MODEL WEIGHTS (SafeTensors Priority)
# ---------------------------------------------------------
print("\n📦 Processing model weights...")

# Try SafeTensors first (preferred)
src_safetensors = os.path.join(SOURCE_DIR, "bad_classifier.safetensors")
dst_safetensors = os.path.join(hf_upload_dir, "model.safetensors")

if os.path.exists(src_safetensors):
    print("   ✅ Found SafeTensors format")
    shutil.copy(src_safetensors, dst_safetensors)
    upload_manifest.append("model.safetensors")
else:
    # Fallback: Convert from PyTorch
    src_pt = os.path.join(SOURCE_DIR, "bad_classifier.pt")
    if os.path.exists(src_pt):
        print("   🔄 Converting PyTorch to SafeTensors...")
        try:
            state_dict = torch.load(src_pt, map_location="cpu")
            save_file(state_dict, dst_safetensors)
            upload_manifest.append("model.safetensors")
            print("   ✅ Conversion successful")
        except Exception as e:
            raise RuntimeError(f"❌ Conversion failed: {e}")
    else:
        raise FileNotFoundError(
            f"❌ No model weights found!\n"
            f"   Checked: bad_classifier.safetensors, bad_classifier.pt"
        )

# ---------------------------------------------------------
# 7. CREATE MODEL ARCHITECTURE FILE
# ---------------------------------------------------------
print("\n📝 Creating model architecture file...")

model_code = f'''"""
BAD (Biased Activation Detection) Classifier
Architecture for {base_model_name}
"""

import torch
import torch.nn as nn

class BADClassifier(nn.Module):
    """
    Binary classifier for detecting biased activations.

    Architecture:
    - Input: Residual stream activation from layer {meta_layer}
    - Hidden: Linear layer with dropout
    - Output: Binary classification (0=biased, 1=unbiased)
    """

    def __init__(self, input_dim={input_dim}, dropout_rate={dropout_rate}):
        super(BADClassifier, self).__init__()
        self.linear = nn.Linear(input_dim, 1)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        """
        Args:
            x: Tensor of shape [batch_size, input_dim]

        Returns:
            logits: Tensor of shape [batch_size, 1]
        """
        x = self.dropout(x)
        logits = self.linear(x)
        return logits

def load_model(weights_path="model.safetensors", device="cpu"):
    """
    Load the BAD classifier from SafeTensors.

    Args:
        weights_path: Path to model.safetensors
        device: Device to load model on

    Returns:
        model: Loaded BADClassifier
    """
    from safetensors.torch import load_file

    model = BADClassifier(input_dim={input_dim}, dropout_rate={dropout_rate})
    state_dict = load_file(weights_path)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    return model
'''

with open(os.path.join(hf_upload_dir, "model.py"), "w") as f:
    f.write(model_code)
upload_manifest.append("model.py")
print("   ✅ model.py created")

# ---------------------------------------------------------
# 8. CREATE INFERENCE EXAMPLE
# ---------------------------------------------------------
print("📝 Creating inference example...")

inference_example = f'''"""
Inference Example for BAD Classifier
"""

import torch
import pickle
import numpy as np
from safetensors.torch import load_file
from model import BADClassifier

def run_inference(activation_vector, model_path=".", device="cpu"):
    """
    Run bias detection on a single activation vector.

    Args:
        activation_vector: numpy array of shape [{input_dim}]
        model_path: Directory containing model files
        device: Device to run inference on

    Returns:
        dict: {{
            'is_biased': bool,
            'bias_probability': float,
            'confidence': float
        }}
    """
    # Load scaler
    with open(f"{{model_path}}/bad_scaler.pkl", "rb") as f:
        scaler = pickle.load(f)

    # Load model
    model = BADClassifier(input_dim={input_dim}, dropout_rate={dropout_rate})
    state_dict = load_file(f"{{model_path}}/model.safetensors")
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()

    # Preprocess
    activation_scaled = scaler.transform(activation_vector.reshape(1, -1))

    # Inference
    with torch.no_grad():
        x = torch.tensor(activation_scaled, dtype=torch.float32, device=device)
        logit = model(x)
        prob_unbiased = torch.sigmoid(logit).item()

    prob_biased = 1 - prob_unbiased
    is_biased = prob_biased > 0.5
    confidence = max(prob_biased, prob_unbiased)

    return {{
        'is_biased': is_biased,
        'bias_probability': prob_biased,
        'confidence': confidence
    }}

# Example usage
if __name__ == "__main__":
    # Dummy activation (replace with actual activation from layer {meta_layer})
    dummy_activation = np.random.randn({input_dim})

    result = run_inference(dummy_activation, model_path=".")

    print(f"Is Biased: {{result['is_biased']}}")
    print(f"Bias Probability: {{result['bias_probability']:.4f}}")
    print(f"Confidence: {{result['confidence']:.4f}}")
'''

with open(os.path.join(hf_upload_dir, "inference.py"), "w") as f:
    f.write(inference_example)
upload_manifest.append("inference.py")
print("   ✅ inference.py created")

# ---------------------------------------------------------
# 9. CREATE REQUIREMENTS.TXT
# ---------------------------------------------------------
print("📝 Creating requirements.txt...")

requirements = '''torch>=2.0.0
safetensors>=0.4.0
numpy>=1.24.0
scikit-learn>=1.3.0
'''

with open(os.path.join(hf_upload_dir, "requirements.txt"), "w") as f:
    f.write(requirements)
upload_manifest.append("requirements.txt")
print("   ✅ requirements.txt created")

# ---------------------------------------------------------
# 10. GENERATE PROFESSIONAL README
# ---------------------------------------------------------
print("📝 Generating model card...")

readme_content = f'''---
license: apache-2.0
library_name: pytorch
tags:
  - bias-detection
  - fairness
  - fairsteer
  - interpretability
pipeline_tag: text-classification
---

# BAD Classifier: Biased Activation Detection

Bias detection classifier trained for **{base_model_name}** using the FairSteer methodology.

## Model Details

- **Base Model**: `{base_model_name}`
- **Target Layer**: {meta_layer}
- **Architecture**: Linear probe with dropout ({meta_arch})
- **Performance**: {meta_score:.2%} balanced accuracy on BBQ benchmark
- **Training Date**: {datetime.now().strftime('%Y-%m-%d')}

## What it Does

This classifier analyzes the internal activations of {base_model_name} at layer {meta_layer} to detect whether the model is exhibiting biased reasoning patterns. It outputs:

- **0 (Biased)**: The activation indicates the model is making decisions based on social biases
- **1 (Unbiased)**: The activation indicates the model is reasoning appropriately

## Usage

### Quick Start
```python
from inference import run_inference
import numpy as np

# Your activation from {base_model_name} layer {meta_layer}
activation = np.random.randn({input_dim})  # Replace with actual activation

result = run_inference(activation, model_path=".")

print(f"Biased: {{result['is_biased']}}")
print(f"Confidence: {{result['confidence']:.2%}}")
```

### Manual Loading
```python
import torch
import pickle
from safetensors.torch import load_file
from model import BADClassifier

# Load scaler
with open("bad_scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# Load model
model = BADClassifier(input_dim={input_dim})
state_dict = load_file("model.safetensors")
model.load_state_dict(state_dict)
model.eval()

# Inference
activation_scaled = scaler.transform(activation.reshape(1, -1))
with torch.no_grad():
    x = torch.tensor(activation_scaled, dtype=torch.float32)
    prob_unbiased = torch.sigmoid(model(x)).item()

is_biased = prob_unbiased < 0.5
```

## Files

- `model.safetensors` - Model weights (SafeTensors format)
- `bad_scaler.pkl` - StandardScaler for preprocessing
- `config.json` - Model configuration and metadata
- `model.py` - Model architecture definition
- `inference.py` - Ready-to-use inference script
- `requirements.txt` - Python dependencies

## Training Details

This model was trained on the BBQ (Bias Benchmark for QA) dataset to distinguish between:
- **Biased activations**: Model predictions driven by social stereotypes
- **Unbiased activations**: Model predictions based on factual reasoning

Training used:
- Binary cross-entropy loss with class balancing
- Mixed precision training (FP16)
- Early stopping based on validation balanced accuracy

## Limitations

- Only works with {base_model_name} at layer {meta_layer}
- Requires exact activation extraction from the specified layer
- Performance may degrade on out-of-distribution biases

## Citation
```bibtex
@article{{fairsteer2024,
  title={{FairSteer: Dynamic Debiasing of Large Language Models}},
  author={{Your Name}},
  journal={{arXiv preprint}},
  year={{2024}}
}}
```

## License

Apache 2.0
'''

with open(os.path.join(hf_upload_dir, "README.md"), "w") as f:
    f.write(readme_content)
upload_manifest.append("README.md")
print("   ✅ README.md created")

# ---------------------------------------------------------
# 11. UPLOAD TO HUGGINGFACE
# ---------------------------------------------------------
print("\n" + "="*80)
print(" 🚀 UPLOADING TO HUGGINGFACE")
print("="*80 + "\n")

api = HfApi()

try:
    # Create repo (private or public)
    create_repo(
        repo_id=REPO_ID,
        token=hf_token,
        private=config.hf_private,
        repo_type="model",
        exist_ok=True
    )
    print(f"✅ Repository ready: {REPO_ID}")

    # Upload
    print(f"\n📤 Uploading {len(upload_manifest)} files...")
    for file in upload_manifest:
        print(f"   • {file}")

    api.upload_folder(
        folder_path=hf_upload_dir,
        repo_id=REPO_ID,
        repo_type="model",
        token=hf_token,
        commit_message=f"Deploy BAD Classifier - Layer {meta_layer} ({meta_score:.2%} acc)"
    )

    print("\n" + "="*80)
    print(" ✅ DEPLOYMENT SUCCESSFUL")
    print("="*80)
    print(f"\n🔗 Model URL: https://huggingface.co/{REPO_ID}")
    print(f"\n📦 Files uploaded:")
    for file in upload_manifest:
        print(f"   ✅ {file}")
    print("\n🎯 Ready for inference and DSV computation!")

except Exception as e:
    print(f"\n❌ Upload failed: {e}")
    import traceback
    traceback.print_exc()
    raise

finally:
    # Cleanup temp directory
    if os.path.exists(hf_upload_dir):
        shutil.rmtree(hf_upload_dir)
        print("\n🧹 Temporary files cleaned up")

print("="*80 + "\n")